# Fraud triage under a review budget: complete pipeline for the DSS manuscript
One notebook, one run: raw data -> models -> calibrated probabilities ->
decision policies -> statistics -> every table, figure and number quoted in
the manuscript. No other notebook is needed as input.

**Inputs:** the IEEE-CIS competition data (`train_transaction.csv`,
`train_identity.csv`) and the ULB `creditcard.csv` as Kaggle datasets.
BankSim is downloaded in Step 2. **Settings:** GPU T4 x2, Internet On,
Save Version -> Save & Run All (about 3 hours).
**Output:** `/kaggle/working/paper_package.zip`.

| Steps | Content |
|---|---|
| 1-8 | data, chronological splits, seven scoring models, FT-Transformer seeds and ablations |
| 9 | decision-layer toolkit (cost model, policies, budget modes) |
| 10-12 | calibration, offline policy comparison with bootstrap and sensitivity, LP check |
| 13-15 | budget enforced on arrival, per-period capacity with adaptive threshold, imperfect analysts |
| 16-17 | cost-scale sweep, calibration sample size |
| 18-20 | tables, number registry, main and additional figures |
| 21-26 | revision analyses: block bootstrap with Holm correction and effect sizes, tuned competitors and decomposition, alternative cost models, distribution shift, additional chronological windows, revision tables |
| 27 | export |

## Step 1: Configuration
Central constants: data paths, output folders, seeds, split ratios, training
settings, decision-layer parameters and figure settings. `note()` records every
number that the manuscript quotes, with its source file.

In [ ]:
# ---- CONFIG ----
import os, gc, json, time, glob, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

_find = lambda name: (glob.glob(f"/kaggle/input/**/{name}", recursive=True) or [None])[0]
DATA_DIR    = os.path.dirname(_find("train_transaction.csv"))          # IEEE-CIS
ULB_CSV     = _find("creditcard.csv")
BANKSIM_CSV = _find("bs140513_032310.csv") or "/kaggle/working/dataset/bs140513_032310.csv"
BANKSIM_URL = ("https://raw.githubusercontent.com/atavci/fraud-detection-on-banksim-data/master/"
               "Data/synthetic-data-from-a-financial-payment-system/bs140513_032310.csv")

WORK = "/kaggle/working"
DIRS = {k: os.path.join(WORK, k) for k in ["fig", "dataset", "output", "model"]}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

SEEDS, BS_SEEDS = [42, 43, 44, 45, 46], [42, 43, 44]
SPLIT       = (0.70, 0.15, 0.15)          # chronological train / val / test
TIME_COL, LABEL, AMOUNT_COL = "TransactionDT", "isFraud", "TransactionAmt"
TOPK_FTT    = 150                         # IEEE-CIS features passed to the FT-Transformer
DROP_COLS   = ["TransactionID", "TransactionDT", "isFraud", "split"]
RFP_BUDGETS = [100, 500, 1000]            # FP budgets for Recall@FP on IEEE-CIS
FTT_BATCH, MAX_EP, PATIENCE = 2048, 40, 4

SCORERS     = ["FTT_ens", "LightGBM", "XGBoost", "CatBoost"]   # scorers fed to the decision layer
COST_RATIOS = [1, 5, 10, 25, 50, 100]
K_GRID = {"ieee": [500, 1000, 2000, 4000, 8000],                   # reviews per test window
          "ulb":  [10, 20, 50, 100, 200, 400],
          "bs":   [50, 100, 200, 400, 800, 1600]}
MID_K  = {t: g[len(g) // 2] for t, g in K_GRID.items()}            # 2000 / 100 / 400
C_REV, C_BLK, THETA_HIGH = 10.0, 50.0, 0.9                         # main cost setting
POL_CALIBS  = ["isotonic", "platt"]       # isotonic = main analysis, Platt = robustness
N_BOOT      = 1000                        # paired bootstrap resamples
SENS_C_REV, SENS_C_BLK = [5, 10, 20], [25, 50, 100]
LAMBDAS     = [0.1, 0.2, 0.5, 1, 2, 5, 10]                          # cost-scale sweep
CAL_SIZE_GRID = {"ieee": [25, 50, 100, 200, 400, 800, 1600, 3042],
                 "bs":   [25, 50, 100, 200, 400, 970],
                 "ulb":  [14, 28, 56]}
N_REP       = 30                          # validation subsamples per calibration size
Q_GRID      = [0.6, 0.7, 0.8, 0.9, 1.0]   # analyst detection probability (imperfect review)
ADAPT_BETA  = 0.5                         # weight of the latest period in the adaptive threshold
PERIOD_NAME = {"ieee": "day", "ulb": "hour", "bs": "day"}   # unit of the per-period capacity

# revision analyses (Steps 21-26)
N_BOOT_BLOCK = 10000                      # moving-block bootstrap resamples
N_BLOCKS     = 50                         # blocks per test window (block = n_test / 50 consecutive transactions)
ALPHA        = 0.05                       # family-wise level for the Holm correction
PRIMARY      = ["binary_bayes_amount", "threeway_expected_loss"]   # pre-specified primary comparisons (P2, P6)
THETA_GRID   = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99]               # theta_H tuned on validation for P5/P6
RECOVERY     = [0.5, 0.75]                # share of a missed fraud amount that is lost
CBLK_FIXED   = 0.5                        # amount-dependent C_blk: this share fixed, the rest proportional to amount
Q_BY_AMOUNT  = (0.95, 0.70)               # analyst accuracy at or below / above the validation median fraud amount
PREV_FACTORS = [0.5, 2.0]                 # fraud prevalence of the test window multiplied by these factors
N_SHIFT      = 30                         # resamples per prevalence factor
ROLL_WINDOWS = {"W1": (0.40, 0.55, 0.70), "W2": (0.55, 0.70, 0.85), "W3": (SPLIT[0], SPLIT[0] + SPLIT[1], 1.00)}   # IEEE-CIS (train end, val end, test end); W3 = main split

DS_NAME  = {"ieee": "IEEE-CIS", "ulb": "ULB", "bs": "BankSim"}
POLICIES = ["binary_F1", "binary_bayes_amount", "topK_review", "reject_option",
            "threeway_fixed", "threeway_expected_loss", "threeway_bayes_budget"]
BUDGETED = POLICIES[2:]
PROPOSED = "threeway_bayes_budget"
POL_SHORT = dict(zip(POLICIES, ["P1", "P2", "P3", "P4", "P5", "P6", "P7"]))
POL_NAME = {"binary_F1": "Binary, F1 threshold (P1)", "binary_bayes_amount": "Binary, Bayes rule (P2)",
            "topK_review": "Top-K review (P3)", "reject_option": "Reject option (P4)",
            "threeway_fixed": "Three-way, fixed (P5)", "threeway_expected_loss": "Three-way, expected loss (P6)",
            "threeway_bayes_budget": "Budgeted Bayes (P7)"}
SC_NAME = {"FTT_ens": "FT-Transformer", "LightGBM": "LightGBM", "XGBoost": "XGBoost", "CatBoost": "CatBoost"}

# figures: DSS full-width artwork, 8 pt text, PDF (vector) + 600 dpi PNG + CSV per figure
FIG_W = 7.0
os.makedirs(f"{DIRS['fig']}/extra", exist_ok=True)
os.makedirs(f"{DIRS['output']}/tables", exist_ok=True)
NUMBERS = []                              # (key, value, source) for every number quoted in the manuscript
def note(key, value, source):
    NUMBERS.append(dict(key=key, value=value, source=source))

print("IEEE-CIS:", DATA_DIR, "\nULB:", ULB_CSV, "\nBankSim:", BANKSIM_CSV)

## Step 2: Data acquisition and integrity check
BankSim is downloaded if it is not attached as input. Row and fraud counts of
all three datasets are checked against the published figures; the run stops
if any check fails.

In [ ]:
import urllib.request
t0 = time.time()
if not os.path.exists(BANKSIM_CSV):
    urllib.request.urlretrieve(BANKSIM_URL, BANKSIM_CSV)

tt = pd.read_csv(f"{DATA_DIR}/train_transaction.csv", usecols=[LABEL, TIME_COL])
n_id = sum(1 for _ in open(f"{DATA_DIR}/train_identity.csv")) - 1
u_ = pd.read_csv(ULB_CSV, usecols=["Class"])
b_ = pd.read_csv(BANKSIM_CSV, usecols=["fraud"])
chk = {"ieee_tx": (len(tt), 590540), "ieee_identity": (n_id, 144233),
       "ieee_fraud": (int(tt[LABEL].sum()), 20663),
       "ulb_tx": (len(u_), 284807), "ulb_fraud": (int(u_.Class.sum()), 492),
       "bs_tx": (len(b_), 594643), "bs_fraud": (int(b_.fraud.sum()), 7200)}
for k, (got, exp) in chk.items():
    print(f"{k:14s} {got:>9,}  expected {exp:>9,}  {'OK' if got == exp else 'MISMATCH'}")
print(f"IEEE-CIS time span: {(tt[TIME_COL].max() - tt[TIME_COL].min()) / 86400:.1f} days | {time.time()-t0:.0f} s")
assert all(g == e for g, e in chk.values()), "dataset integrity check failed"
del tt, u_, b_; gc.collect()

## Step 3: IEEE-CIS load, merge and chronological split
`train_transaction` is left-joined with `train_identity` on `TransactionID`.
Numeric columns are downcast to reduce memory. Rows are sorted by
`TransactionDT` and split 70/15/15 in time order. The unlabelled competition
test files are not used.

In [ ]:
t0 = time.time()
def downcast(df):
    for c in df.select_dtypes("float64").columns:
        df[c] = pd.to_numeric(df[c], downcast="float")
    for c in df.select_dtypes("int64").columns:
        df[c] = pd.to_numeric(df[c], downcast="integer")
    return df

tr = downcast(pd.read_csv(f"{DATA_DIR}/train_transaction.csv"))
ti = downcast(pd.read_csv(f"{DATA_DIR}/train_identity.csv"))
df = tr.merge(ti, on="TransactionID", how="left")
del tr, ti; gc.collect()

df = df.sort_values(TIME_COL, kind="mergesort").reset_index(drop=True)
n = len(df)
a, b = int(n * SPLIT[0]), int(n * (SPLIT[0] + SPLIT[1]))
df["split"] = np.where(np.arange(n) < a, "train", np.where(np.arange(n) < b, "val", "test"))

summary = (df.groupby("split", sort=False)
             .agg(n=(LABEL, "size"), fraud=(LABEL, "sum"),
                  t_start=(TIME_COL, "min"), t_end=(TIME_COL, "max")))
summary["fraud_rate_%"] = 100 * summary.fraud / summary.n
summary["days_from_start"] = (summary.t_start - df[TIME_COL].min()) / 86400
print(summary)
print(f"merged shape: {df.shape} | identity coverage: {df['id_01'].notna().mean():.2%} | "
      f"memory: {df.memory_usage(deep=True).sum()/1e9:.2f} GB | {time.time()-t0:.0f} s")
summary.to_csv(f"{DIRS['output']}/ieee_split_summary.csv")
_f = df[LABEL] == 1
note("ieee_median_fraud_amount_all", float(df.loc[_f, AMOUNT_COL].median()), "train_transaction.csv")
note("ieee_median_legit_amount_all", float(df.loc[~_f, AMOUNT_COL].median()), "train_transaction.csv")
note("ieee_identity_coverage", float(df["id_01"].notna().mean()), "train_identity.csv")
note("ieee_test_window_days", float((summary.t_end["test"] - summary.t_start["test"]) / 86400), "ieee_split_summary.csv")
np.savez_compressed(f"{DIRS['dataset']}/ieee_meta.npz", t=df[TIME_COL].values, y=df[LABEL].values,
                    amt=df[AMOUNT_COL].values, split=df["split"].values.astype("U5"))

## Step 4: IEEE-CIS encoding, missing values and feature selection
* Categorical columns are frequency-encoded with training-partition
  frequencies; missing values form their own category, unseen categories map to 0.
* `TransactionDT` is used only for splitting, never as a feature.
* Tree models receive all features with native missing-value handling.
* The FT-Transformer receives the top-150 features by LightGBM gain (LightGBM
  fitted on train, early-stopped on validation; test untouched), standardized
  with training statistics, mean-imputed (0 after scaling) and clipped to [-10, 10].
  The top-150 features retain about 92% of the cumulative gain; the full
  cumulative-gain curve is saved for reporting.

In [ ]:
import lightgbm as lgb
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
t0 = time.time()

m_tr, m_va, m_te = (df["split"] == s for s in ("train", "val", "test"))
cat_cols  = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c]) and c not in DROP_COLS]
feat_cols = [c for c in df.columns if c not in DROP_COLS]
for c in cat_cols:
    s = df[c].fillna("__nan__")
    df[c] = s.map(s[m_tr].value_counts(normalize=True)).fillna(0).astype("float32")

X   = df[feat_cols].astype("float32").values
y   = df[LABEL].values.astype("int8")
amt = df[AMOUNT_COL].values.astype("float32")
Xtr, Xva, Xte = X[m_tr.values], X[m_va.values], X[m_te.values]
ytr, yva, yte = y[m_tr.values], y[m_va.values], y[m_te.values]
print(f"features: {len(feat_cols)} ({len(cat_cols)} categorical) | "
      f"train {Xtr.shape}, val {Xva.shape}, test {Xte.shape}")

sel = lgb.LGBMClassifier(n_estimators=3000, learning_rate=0.05, num_leaves=127, subsample=0.8,
                         subsample_freq=1, colsample_bytree=0.5, random_state=42, n_jobs=-1, verbose=-1)
sel.fit(Xtr, ytr, eval_set=[(Xva, yva)], eval_metric="average_precision",
        callbacks=[lgb.early_stopping(100, verbose=False)])
imp = (pd.DataFrame({"feature": feat_cols, "gain": sel.booster_.feature_importance("gain")})
         .sort_values("gain", ascending=False).reset_index(drop=True))
imp.to_csv(f"{DIRS['output']}/ieee_feature_importance.csv", index=False)
cum = (imp.gain.cumsum() / imp.gain.sum()).values
pd.DataFrame({"k": np.arange(1, len(cum) + 1), "cum_gain_share": cum}) \
  .to_csv(f"{DIRS['output']}/ieee_cumulative_gain.csv", index=False)
top = imp.feature[:TOPK_FTT].tolist()
json.dump(top, open(f"{DIRS['output']}/ieee_selected_features.json", "w"), indent=1)
print(f"selector: best_iter={sel.best_iteration_} | val AUPRC="
      f"{average_precision_score(yva, sel.predict_proba(Xva)[:, 1]):.4f}")
print("cumulative gain share:", {k: round(float(cum[k - 1]), 3) for k in [100, 125, 150, 175, 200, 250] if k <= len(cum)})

np.savez_compressed(f"{DIRS['dataset']}/ieee_tree.npz", Xtr=Xtr, Xva=Xva, Xte=Xte,
                    ytr=ytr, yva=yva, yte=yte, amt_va=amt[m_va.values], amt_te=amt[m_te.values],
                    feats=np.array(feat_cols))
idx = [feat_cols.index(f) for f in top]
sc = StandardScaler().fit(Xtr[:, idx])
prep = lambda A: np.clip(np.nan_to_num(sc.transform(A[:, idx]), nan=0.0), -10, 10).astype("float32")
np.savez_compressed(f"{DIRS['dataset']}/ieee_ftt.npz", Xtr=prep(Xtr), Xva=prep(Xva), Xte=prep(Xte),
                    ytr=ytr, yva=yva, yte=yte, amt_te=amt[m_te.values], feats=np.array(top))
day = ((df[TIME_COL] - df[TIME_COL].min()) // 86400).astype(int).values
np.savez_compressed(f"{DIRS['dataset']}/ieee_decision.npz", yva=yva, yte=yte,
                    amt_va=amt[m_va.values], amt_te=amt[m_te.values],
                    per_va=day[m_va.values], per_te=day[m_te.values])
note("ieee_gain_share_top150", float(cum[TOPK_FTT - 1]), "ieee_cumulative_gain.csv")
print(f"K={TOPK_FTT} | gain share={cum[TOPK_FTT-1]:.3f} | {time.time()-t0:.0f} s")
del df, X, Xtr, Xva, Xte; gc.collect()

## Step 5: IEEE-CIS baseline models
Every model is trained on the training partition, early-stopped (where
applicable) and thresholded on validation (F1-maximizing threshold), and
evaluated once on test. Tree ensembles use all 431 features with native
missing-value handling; logistic regression and the MLP use the same 150
standardized features as the FT-Transformer.

In [ ]:
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score, precision_recall_curve, confusion_matrix

T  = np.load(f"{DIRS['dataset']}/ieee_tree.npz")
Fz = np.load(f"{DIRS['dataset']}/ieee_ftt.npz")
yva, yte = T["yva"], T["yte"]

def recall_at_fp(y, p, B):
    o = np.argsort(-p); ys = y[o]
    fp = np.cumsum(ys == 0); tp = np.cumsum(ys == 1); ok = fp <= B
    return (tp[ok].max() if ok.any() else 0) / y.sum()

def evaluate(name, p_va, p_te, secs):
    pr, rc, th = precision_recall_curve(yva, p_va)
    t = float(th[np.argmax((2 * pr * rc / np.maximum(pr + rc, 1e-12))[:-1])])
    tn, fp, fn, tp = confusion_matrix(yte, (p_te >= t).astype(int)).ravel()
    row = dict(model=name, AUPRC=average_precision_score(yte, p_te), AUROC=roc_auc_score(yte, p_te),
               theta=t, TP=tp, FP=fp, FN=fn, Recall=tp / (tp + fn), Precision=tp / max(tp + fp, 1),
               **{f"R@FP{B}": recall_at_fp(yte, p_te, B) for B in RFP_BUDGETS}, train_s=round(secs))
    np.savez(f"{DIRS['output']}/pred_ieee_{name}.npz", p_va=p_va, p_te=p_te)
    print(f"{name:13s} AUPRC={row['AUPRC']:.4f} AUROC={row['AUROC']:.4f} "
          f"Rec={row['Recall']:.3f} Prec={row['Precision']:.3f} ({row['train_s']} s)")
    return row

rows = []
def run(name, fit_predict):
    t0 = time.time(); p_va, p_te = fit_predict()
    rows.append(evaluate(name, p_va, p_te, time.time() - t0)); gc.collect()

def f_lr():
    m = LogisticRegression(max_iter=2000, class_weight="balanced").fit(Fz["Xtr"], Fz["ytr"])
    return m.predict_proba(Fz["Xva"])[:, 1], m.predict_proba(Fz["Xte"])[:, 1]
def f_mlp():
    m = MLPClassifier(hidden_layer_sizes=(128, 64), alpha=1e-4, batch_size=512, learning_rate_init=1e-3,
                      max_iter=30, early_stopping=True, n_iter_no_change=6, random_state=42).fit(Fz["Xtr"], Fz["ytr"])
    return m.predict_proba(Fz["Xva"])[:, 1], m.predict_proba(Fz["Xte"])[:, 1]
spw = (T["ytr"] == 0).sum() / T["ytr"].sum()
def f_rf():
    m = RandomForestClassifier(n_estimators=150, min_samples_leaf=5, n_jobs=-1,
                               class_weight="balanced_subsample", random_state=42).fit(T["Xtr"], T["ytr"])
    return m.predict_proba(T["Xva"])[:, 1], m.predict_proba(T["Xte"])[:, 1]
def f_xgb():
    m = xgb.XGBClassifier(n_estimators=3000, max_depth=6, learning_rate=0.05, subsample=0.9,
                          colsample_bytree=0.9, scale_pos_weight=spw, tree_method="hist", device="cuda",
                          eval_metric="aucpr", early_stopping_rounds=100, random_state=42)
    m.fit(T["Xtr"], T["ytr"], eval_set=[(T["Xva"], yva)], verbose=False)
    return m.predict_proba(T["Xva"])[:, 1], m.predict_proba(T["Xte"])[:, 1]
def f_lgb():
    m = lgb.LGBMClassifier(n_estimators=3000, num_leaves=31, learning_rate=0.05,
                           random_state=42, n_jobs=-1, verbose=-1)
    m.fit(T["Xtr"], T["ytr"], eval_set=[(T["Xva"], yva)], eval_metric="average_precision",
          callbacks=[lgb.early_stopping(100, verbose=False)])
    return m.predict_proba(T["Xva"])[:, 1], m.predict_proba(T["Xte"])[:, 1]
def f_cat():
    m = CatBoostClassifier(iterations=3000, depth=6, learning_rate=0.05, task_type="GPU",
                           eval_metric="Logloss", early_stopping_rounds=100, random_seed=42, verbose=0)
    m.fit(T["Xtr"], T["ytr"], eval_set=(T["Xva"], yva), use_best_model=True)
    return m.predict_proba(T["Xva"])[:, 1], m.predict_proba(T["Xte"])[:, 1]

for name, f in [("LogReg", f_lr), ("MLP", f_mlp), ("RandomForest", f_rf),
                ("XGBoost", f_xgb), ("LightGBM", f_lgb), ("CatBoost", f_cat)]:
    run(name, f)
pd.DataFrame(rows).to_csv(f"{DIRS['output']}/ieee_phase1_results.csv", index=False)
print(pd.DataFrame(rows).round(4).to_string(index=False))

## Step 6: IEEE-CIS FT-Transformer (5 seeds)
Feature tokenizer with a learnable [CLS] token, d = 24, two pre-norm
transformer blocks with two attention heads, alpha-balanced focal loss
(gamma = 2, alpha = 0.75), square-root-balanced class weights, Adam (1e-3),
early stopping on validation AUPRC (patience 4, at most 40 epochs). Input:
the 150 standardized features of Step 4. The deployed score is the five-seed
ensemble mean.

In [ ]:
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import tensorflow as tf
from tensorflow.keras import layers
print("TF", tf.__version__, "| GPUs:", len(tf.config.list_physical_devices("GPU")))

def build_ftt(n_feat, d=24, n_layers=2, n_heads=2, drop=0.1):
    class FT(layers.Layer):
        def build(self, s):
            f = s[-1]
            self.W   = self.add_weight(name="W", shape=(f, d), initializer="glorot_uniform")
            self.b   = self.add_weight(name="b", shape=(f, d), initializer="zeros")
            self.cls = self.add_weight(name="cls", shape=(1, 1, d), initializer="glorot_uniform")
        def call(self, x):
            tok = x[..., None] * self.W + self.b
            return tf.concat([tf.repeat(self.cls, tf.shape(x)[0], axis=0), tok], axis=1)
    def block(x):
        h = layers.LayerNormalization()(x)
        h = layers.MultiHeadAttention(num_heads=n_heads, key_dim=d // n_heads, dropout=drop)(h, h)
        x = layers.Add()([x, h]); h = layers.LayerNormalization()(x)
        h = layers.Dense(d * 4, activation="gelu")(h); h = layers.Dropout(drop)(h); h = layers.Dense(d)(h)
        return layers.Add()([x, h])
    inp = layers.Input(shape=(n_feat,)); x = FT()(inp)
    for _ in range(n_layers): x = block(x)
    x = layers.LayerNormalization()(x)
    return tf.keras.Model(inp, layers.Dense(1, activation="sigmoid")(x[:, 0, :]))

def focal(g=2., a=.75):
    def loss(y, p):
        p = tf.clip_by_value(p, 1e-7, 1 - 1e-7); pt = y * p + (1 - y) * (1 - p)
        al = y * a + (1 - y) * (1 - a)
        return tf.reduce_mean(-al * tf.pow(1 - pt, g) * tf.math.log(pt))
    return loss

Xtr_f, Xva_f, Xte_f = Fz["Xtr"], Fz["Xva"], Fz["Xte"]
ytr_f, yva_f = Fz["ytr"].astype("float32"), Fz["yva"].astype("float32")
cw = {0: 1.0, 1: float(((ytr_f == 0).sum() / ytr_f.sum()) ** 0.5)}
log = []
for s in SEEDS:
    t0 = time.time(); tf.keras.backend.clear_session(); np.random.seed(s); tf.random.set_seed(s)
    m = build_ftt(Xtr_f.shape[1])
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=focal(),
              metrics=[tf.keras.metrics.AUC(curve="PR", name="pr_auc")])
    h = m.fit(Xtr_f, ytr_f, validation_data=(Xva_f, yva_f), epochs=MAX_EP, batch_size=FTT_BATCH,
              class_weight=cw, verbose=0,
              callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_pr_auc", mode="max",
                         patience=PATIENCE, restore_best_weights=True)])
    hist = h.history["val_pr_auc"]; best_ep = int(np.argmax(hist)) + 1
    p_va = m.predict(Xva_f, batch_size=8192, verbose=0).ravel()
    p_te = m.predict(Xte_f, batch_size=8192, verbose=0).ravel()
    np.savez(f"{DIRS['output']}/pred_ieee_FTT_seed{s}.npz", p_va=p_va, p_te=p_te, best_val=max(hist))
    m.save_weights(f"{DIRS['model']}/ftt_ieee_seed{s}.weights.h5")
    ceiling = len(hist) == MAX_EP and best_ep > MAX_EP - PATIENCE   # stopped by the epoch cap, not by patience
    log.append(dict(seed=s, best_val_auprc=max(hist), best_epoch=best_ep, epochs_run=len(hist),
                    test_auprc=average_precision_score(yte, p_te), minutes=(time.time() - t0) / 60,
                    hit_ceiling=ceiling))
    print(f"[seed {s}] val AUPRC={max(hist):.4f} (epoch {best_ep}/{len(hist)}) "
          f"test AUPRC={log[-1]['test_auprc']:.4f} | {log[-1]['minutes']:.1f} min"
          + ("  *** STOPPED AT EPOCH CAP ***" if ceiling else ""))

L = pd.DataFrame(log); L.to_csv(f"{DIRS['output']}/ieee_ftt_training_log.csv", index=False)
P = [np.load(f"{DIRS['output']}/pred_ieee_FTT_seed{s}.npz") for s in SEEDS]
rows.append(evaluate("FTT_ens", np.mean([x["p_va"] for x in P], 0), np.mean([x["p_te"] for x in P], 0),
                     L.minutes.sum() * 60))
pd.DataFrame(rows).to_csv(f"{DIRS['output']}/ieee_phase1_results.csv", index=False)
print(f"seed-level test AUPRC {L.test_auprc.mean():.4f} ± {L.test_auprc.std():.4f}")

## Step 7: ULB and BankSim preparation
Original protocol: chronological 70/15/15 split (ULB by `Time`, BankSim by
`step`); scaler and all categorical statistics fitted on the training
partition only. Validation and test amounts are kept for the amount-aware
decision layer.

In [ ]:
t0 = time.time()
u = pd.read_csv(ULB_CSV).sort_values("Time", kind="mergesort").reset_index(drop=True)
n = len(u); a_, b_ = int(n * .70), int(n * .85)
feat_u = [c for c in u.columns if c != "Class"]
sc_u = StandardScaler().fit(u.iloc[:a_][feat_u])
parts = {"tr": u.iloc[:a_], "va": u.iloc[a_:b_], "te": u.iloc[b_:]}
np.savez_compressed(f"{DIRS['dataset']}/ulb.npz",
    **{f"X{k}": sc_u.transform(v[feat_u]).astype("float32") for k, v in parts.items()},
    **{f"y{k}": v.Class.values.astype("int8") for k, v in parts.items()},
    amt_va=parts["va"].Amount.values.astype("float32"), amt_te=parts["te"].Amount.values.astype("float32"))
np.savez_compressed(f"{DIRS['dataset']}/ulb_decision.npz",
    yva=parts["va"].Class.values.astype("int8"), yte=parts["te"].Class.values.astype("int8"),
    amt_va=parts["va"].Amount.values.astype("float32"), amt_te=parts["te"].Amount.values.astype("float32"),
    per_va=(parts["va"].Time.values // 3600).astype(int), per_te=(parts["te"].Time.values // 3600).astype(int))
f_ = u.Class == 1
for scope, uu in [("all", u), ("test", parts["te"])]:
    ff = uu.Class == 1
    note(f"ulb_{scope}_share_fraud_amount_le1", float((uu.Amount[ff] <= 1).mean()), "creditcard.csv")
    note(f"ulb_{scope}_share_fraud_amount_eq0", float((uu.Amount[ff] == 0).mean()), "creditcard.csv")
    note(f"ulb_{scope}_median_fraud_amount", float(uu.Amount[ff].median()), "creditcard.csv")
    note(f"ulb_{scope}_median_legit_amount", float(uu.Amount[~ff].median()), "creditcard.csv")
print("ULB    ", {k: (len(v), int(v.Class.sum())) for k, v in parts.items()},
      f"| frauds with amount <= 1: {(u.Amount[f_] <= 1).mean():.3f}")

d = pd.read_csv(BANKSIM_CSV)
for c in ["customer", "age", "gender", "merchant", "category"]:
    d[c] = d[c].astype(str).str.strip("'")
d = d.sort_values("step", kind="mergesort").reset_index(drop=True)
n = len(d); a_, b_ = int(n * .70), int(n * .85)
trd = d.iloc[:a_]
d["merch_freq"] = d.merchant.map(trd.merchant.value_counts(normalize=True)).fillna(0)
d["cust_freq"]  = d.customer.map(trd.customer.value_counts(normalize=True)).fillna(0)
d["cat_risk"]   = d.category.map(trd.groupby("category").fraud.mean()).fillna(trd.fraud.mean())
Xb = pd.get_dummies(d[["age", "gender", "category"]], drop_first=False)
for c in ["amount", "step", "merch_freq", "cust_freq", "cat_risk"]:
    Xb[c] = d[c]
Xb = Xb.astype("float32"); yb = d.fraud.values.astype("int8")
sl = {"tr": slice(0, a_), "va": slice(a_, b_), "te": slice(b_, n)}
sc_b = StandardScaler().fit(Xb.iloc[sl["tr"]])
np.savez_compressed(f"{DIRS['dataset']}/banksim.npz",
    **{f"X{k}": sc_b.transform(Xb.iloc[s]).astype("float32") for k, s in sl.items()},
    **{f"y{k}": yb[s] for k, s in sl.items()},
    amt_va=d.amount.values[sl["va"]].astype("float32"), amt_te=d.amount.values[sl["te"]].astype("float32"))
np.savez_compressed(f"{DIRS['dataset']}/bs_decision.npz", yva=yb[sl["va"]], yte=yb[sl["te"]],
    amt_va=d.amount.values[sl["va"]].astype("float32"), amt_te=d.amount.values[sl["te"]].astype("float32"),
    per_va=d.step.values[sl["va"]].astype(int), per_te=d.step.values[sl["te"]].astype(int))
for t_, am, yy in [("ulb", u.Amount.values, u.Class.values), ("bs", d.amount.values, d.fraud.values)]:
    note(f"{t_}_median_fraud_amount_all", float(np.median(am[yy == 1])), "raw csv")
    note(f"{t_}_median_legit_amount_all", float(np.median(am[yy == 0])), "raw csv")
print("BankSim", {k: (s.stop - s.start, int(yb[s].sum())) for k, s in sl.items()}, "| features:", Xb.shape[1])
print(f"{time.time()-t0:.0f} s")
del u, d, Xb; gc.collect()

## Step 8: ULB and BankSim models and ULB ablations
Same six baselines and FT-Transformer protocol as IEEE-CIS (ULB: five seeds,
batch 1,024; BankSim: three seeds, batch 2,048; at most 40 epochs). ULB
ablations (weighted cross-entropy; one transformer block) use three seeds each.

In [ ]:
def r_at_fp(y, p, B):
    o = np.argsort(-p); ys = y[o]; fp = np.cumsum(ys == 0); tp = np.cumsum(ys == 1)
    ok = fp <= B; return (tp[ok].max() if ok.any() else 0) / y.sum()

def evaluate_ds(tag, name, D, p_va, p_te, secs, budgets):
    pr, rc, th = precision_recall_curve(D["yva"], p_va)
    t = float(th[np.argmax((2 * pr * rc / np.maximum(pr + rc, 1e-12))[:-1])])
    tn, fp, fn, tp = confusion_matrix(D["yte"], (p_te >= t).astype(int)).ravel()
    np.savez(f"{DIRS['output']}/pred_{tag}_{name}.npz", p_va=p_va, p_te=p_te)
    row = dict(model=name, AUPRC=average_precision_score(D["yte"], p_te), AUROC=roc_auc_score(D["yte"], p_te),
               theta=t, TP=tp, FP=fp, FN=fn, Recall=tp / (tp + fn), Precision=tp / max(tp + fp, 1),
               **{f"R@FP{B}": r_at_fp(D["yte"], p_te, B) for B in budgets}, train_s=round(secs))
    print(f"[{tag}] {name:13s} AUPRC={row['AUPRC']:.4f} AUROC={row['AUROC']:.4f} ({row['train_s']} s)")
    return row

def train_ftt(tag, D, seed, batch, loss="focal", n_layers=2, name="FTT"):
    t0 = time.time(); tf.keras.backend.clear_session(); np.random.seed(seed); tf.random.set_seed(seed)
    ytr_ = D["ytr"].astype("float32"); yva_ = D["yva"].astype("float32")
    m = build_ftt(D["Xtr"].shape[1], n_layers=n_layers)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss=focal() if loss == "focal" else "binary_crossentropy",
              metrics=[tf.keras.metrics.AUC(curve="PR", name="pr_auc")])
    cw_ = {0: 1.0, 1: float(((ytr_ == 0).sum() / ytr_.sum()) ** 0.5)}
    h = m.fit(D["Xtr"], ytr_, validation_data=(D["Xva"], yva_), epochs=MAX_EP, batch_size=batch,
              class_weight=cw_, verbose=0,
              callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_pr_auc", mode="max",
                         patience=PATIENCE, restore_best_weights=True)])
    hist = h.history["val_pr_auc"]; be = int(np.argmax(hist)) + 1
    p_va = m.predict(D["Xva"], batch_size=8192, verbose=0).ravel()
    p_te = m.predict(D["Xte"], batch_size=8192, verbose=0).ravel()
    np.savez(f"{DIRS['output']}/pred_{tag}_{name}_seed{seed}.npz", p_va=p_va, p_te=p_te, best_val=max(hist))
    ceil = len(hist) == MAX_EP and be > MAX_EP - PATIENCE
    print(f"[{tag}] {name} seed {seed}: val {max(hist):.4f} (ep {be}/{len(hist)}) "
          f"test {average_precision_score(D['yte'], p_te):.4f} | {(time.time()-t0)/60:.1f} min"
          + ("  *** STOPPED AT EPOCH CAP ***" if ceil else ""))
    return dict(tag=tag, name=name, seed=seed, best_val=max(hist), best_epoch=be,
                epochs_run=len(hist), test_auprc=average_precision_score(D["yte"], p_te), ceiling=ceil)

ftt_log = []
for tag, path, seeds, batch, budgets in [("ulb", "ulb.npz", SEEDS, 1024, [20, 50]),
                                         ("bs", "banksim.npz", BS_SEEDS, 2048, [100, 500])]:
    D = np.load(f"{DIRS['dataset']}/{path}"); rows_ds = []
    spw_ = (D["ytr"] == 0).sum() / D["ytr"].sum()
    pp = lambda m: (m.predict_proba(D["Xva"])[:, 1], m.predict_proba(D["Xte"])[:, 1])
    def fit_xgb():
        m = xgb.XGBClassifier(n_estimators=3000, max_depth=6, learning_rate=0.05, subsample=0.9,
                              colsample_bytree=0.9, scale_pos_weight=spw_, tree_method="hist", device="cuda",
                              eval_metric="aucpr", early_stopping_rounds=100, random_state=42)
        m.fit(D["Xtr"], D["ytr"], eval_set=[(D["Xva"], D["yva"])], verbose=False); return pp(m)
    def fit_lgb():
        m = lgb.LGBMClassifier(n_estimators=3000, num_leaves=31, learning_rate=0.05, random_state=42,
                               n_jobs=-1, verbose=-1)
        m.fit(D["Xtr"], D["ytr"], eval_set=[(D["Xva"], D["yva"])], eval_metric="average_precision",
              callbacks=[lgb.early_stopping(100, verbose=False)]); return pp(m)
    def fit_cat():
        m = CatBoostClassifier(iterations=3000, depth=6, learning_rate=0.05, task_type="GPU",
                               eval_metric="Logloss", early_stopping_rounds=100, random_seed=42, verbose=0)
        m.fit(D["Xtr"], D["ytr"], eval_set=(D["Xva"], D["yva"]), use_best_model=True); return pp(m)
    models = [
        ("LogReg", lambda: pp(LogisticRegression(max_iter=2000, class_weight="balanced").fit(D["Xtr"], D["ytr"]))),
        ("MLP", lambda: pp(MLPClassifier(hidden_layer_sizes=(128, 64), alpha=1e-4, batch_size=512,
                                         learning_rate_init=1e-3, max_iter=30, early_stopping=True,
                                         n_iter_no_change=6, random_state=42).fit(D["Xtr"], D["ytr"]))),
        ("RandomForest", lambda: pp(RandomForestClassifier(n_estimators=150, n_jobs=-1,
                                     class_weight="balanced_subsample", random_state=42).fit(D["Xtr"], D["ytr"]))),
        ("XGBoost", fit_xgb), ("LightGBM", fit_lgb), ("CatBoost", fit_cat)]
    for name, f in models:
        t0 = time.time(); pv_, pt_ = f()
        rows_ds.append(evaluate_ds(tag, name, D, pv_, pt_, time.time() - t0, budgets))
    for s in seeds:
        ftt_log.append(train_ftt(tag, D, s, batch))
    P = [np.load(f"{DIRS['output']}/pred_{tag}_FTT_seed{s}.npz") for s in seeds]
    rows_ds.append(evaluate_ds(tag, "FTT_ens", D, np.mean([x["p_va"] for x in P], 0),
                               np.mean([x["p_te"] for x in P], 0), 0, budgets))
    if tag == "ulb":
        for s in [42, 43, 44]:
            ftt_log.append(train_ftt(tag, D, s, batch, loss="bce", name="ABL-bceL2"))
            ftt_log.append(train_ftt(tag, D, s, batch, n_layers=1, name="ABL-focalL1"))
    pd.DataFrame(rows_ds).to_csv(f"{DIRS['output']}/{tag}_phase1_results.csv", index=False)
    print(pd.DataFrame(rows_ds).round(4).to_string(index=False), "\n")
pd.DataFrame(ftt_log).to_csv(f"{DIRS['output']}/ulb_bs_ftt_training_log.csv", index=False)

## Step 9: Decision-layer toolkit
Functions shared by Steps 10 to 17, and a cache of calibrated probabilities.

* **Cost model.** A blocked legitimate transaction costs `C_blk`, a review costs
  `C_rev`, an approved fraud costs its amount. With an imperfect analyst who
  detects a reviewed fraud with probability `q`, a reviewed fraud also costs
  `(1 - q) * amount` in expectation (`q = 1` is the main setting).
* **Policies.** Each policy is split into a fixed block rule, a set of review
  candidates and a review priority. The budget only decides which candidates
  are reviewed, so every policy can be run in every budget mode:
  * `offline`: the K best candidates of the whole test window (hindsight);
  * `online_uncapped` / `online_capped`: a threshold learned on validation is
    applied as transactions arrive, without / with a hard cap of K;
  * `period_hindsight`: a capacity of K/P per period (day or hour), best
    candidates of each period;
  * `period_static` / `period_adaptive`: per-period capacity enforced on
    arrival with a validation threshold that is fixed / updated after every
    period towards the threshold that would have used that period's capacity.
* All calibrators, F1 thresholds and review thresholds use validation data only.

In [ ]:
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import brier_score_loss
from scipy.optimize import minimize_scalar

DEC = {t: dict(np.load(f"{DIRS['dataset']}/{t}_decision.npz")) for t in ["ieee", "ulb", "bs"]}
SEEDS_OF = {"ieee": SEEDS, "ulb": SEEDS, "bs": BS_SEEDS}

# ---------- calibration
def ece(y, p, b=15):
    e, ed = 0.0, np.linspace(0, 1, b + 1)
    for lo, hi in zip(ed[:-1], ed[1:]):
        m = (p >= lo) & ((p < hi) if hi < 1 else (p <= hi))
        if m.any(): e += m.mean() * abs(y[m].mean() - p[m].mean())
    return e
lgt = lambda p: np.log(np.clip(p, 1e-7, 1 - 1e-7) / (1 - np.clip(p, 1e-7, 1 - 1e-7)))
def platt(pv, yv, pt):
    m = LogisticRegression().fit(lgt(pv)[:, None], yv); return m.predict_proba(lgt(pt)[:, None])[:, 1]
def temp(pv, yv, pt):
    zv = lgt(pv); sig = lambda z, T_: 1 / (1 + np.exp(-z / T_))
    nll = lambda T_: -np.mean(yv * np.log(np.clip(sig(zv, T_), 1e-7, 1)) + (1 - yv) * np.log(np.clip(1 - sig(zv, T_), 1e-7, 1)))
    return sig(lgt(pt), minimize_scalar(nll, bounds=(0.05, 20), method="bounded").x)
iso = lambda pv, yv, pt: IsotonicRegression(out_of_bounds="clip").fit(pv, yv).predict(pt)
CAL_FN = {"isotonic": iso, "platt": platt}

def f1_threshold(y, p):
    pr, rc, th = precision_recall_curve(y, p)
    return float(th[np.argmax((2 * pr * rc / np.maximum(pr + rc, 1e-12))[:-1])])

def load_pred(tag, name):
    if name == "FTT_ens":
        d = [np.load(f"{DIRS['output']}/pred_{tag}_FTT_seed{s}.npz") for s in SEEDS_OF[tag]]
        return np.mean([x["p_va"] for x in d], 0), np.mean([x["p_te"] for x in d], 0)
    d = np.load(f"{DIRS['output']}/pred_{tag}_{name}.npz"); return d["p_va"], d["p_te"]

# CAL[tag][scorer][calib] = dict(pv=calibrated validation, pt=calibrated test, th=F1 threshold)
CAL = {}
for t in DEC:
    CAL[t] = {}
    for sc in SCORERS:
        pv, pt = load_pred(t, sc); yv = DEC[t]["yva"]; CAL[t][sc] = {"raw": dict(pv=pv, pt=pt)}
        for cb, fn in CAL_FN.items():
            pvc = fn(pv, yv, pv); CAL[t][sc][cb] = dict(pv=pvc, pt=fn(pv, yv, pt), th=f1_threshold(yv, pvc))

# ---------- policies: fixed block rule, review candidates, review priority
def components(name, p, a, th, c_rev=C_REV, c_blk=C_BLK, q=1.0):
    none, allm = np.zeros_like(p, bool), np.ones_like(p, bool)
    if name == "binary_F1":           return p >= th, none, p
    if name == "binary_bayes_amount": return p * a >= (1 - p) * c_blk, none, p
    if name == "topK_review":         return none, allm, p
    if name == "reject_option":       return p >= th, allm, -np.abs(p - th)
    if name in ("threeway_fixed", "threeway_expected_loss"):
        blk = p >= THETA_HIGH; return blk, ~blk, (p if name == "threeway_fixed" else p * a)
    # budgeted Bayes; q < 1 adds the expected loss of frauds the analyst misses
    c_app, c_bl = p * a, (1 - p) * c_blk
    gain = np.minimum(c_app, c_bl) - c_rev - (1 - q) * c_app
    return c_bl < c_app, gain > 0, gain

def top_k(prio, cand, k):
    idx = np.where(cand)[0]; sel = np.zeros_like(cand)
    sel[idx[np.argsort(-prio[idx], kind="stable")[:max(int(k), 0)]]] = True; return sel

def kth_value(prio, cand, k):
    """Priority of the k-th best candidate: the threshold that would review exactly k."""
    s = np.sort(prio[cand])[::-1]
    return np.inf if (k <= 0 or len(s) == 0) else s[min(int(k), len(s)) - 1]

def select(mode, prio, cand, K, tau=None, per=None, k_per=None, tau0=None, beta=ADAPT_BETA):
    if mode == "offline":          return top_k(prio, cand, K)
    if mode == "online_uncapped":  return cand & (prio >= tau)
    if mode == "online_capped":
        sel = np.zeros_like(cand); sel[np.where(cand & (prio >= tau))[0][:K]] = True; return sel
    sel = np.zeros_like(cand); tau_ = tau0
    for d in np.unique(per):                       # periods in time order
        idx = np.where(per == d)[0]; c = cand[idx]; v = prio[idx]
        if mode == "period_hindsight":
            sel[idx[top_k(v, c, k_per)]] = True; continue
        hit = np.where(c & (v >= tau_))[0][:k_per]; sel[idx[hit]] = True
        if mode == "period_adaptive" and c.any():
            tau_ = (1 - beta) * tau_ + beta * kth_value(v, c, k_per) if np.isfinite(tau_) else kth_value(v, c, k_per)
    return sel

def tx_cost(blk, rev, y1, a, c_rev=C_REV, c_blk=C_BLK, q=1.0):
    """Realised per-transaction cost (expectation over analyst misses when q < 1)."""
    return c_blk * (blk & ~y1) + c_rev * rev + a * (~blk & ~rev & y1) + (1 - q) * a * (rev & y1)

def decide(name, t, sc, calib="isotonic", K=None, mode="offline", c_rev=C_REV, c_blk=C_BLK, q=1.0, q_policy=None):
    """Block and review masks of a policy on the test window. q_policy: analyst accuracy assumed by P7."""
    C = CAL[t][sc][calib]; D = DEC[t]; qp = q if q_policy is None else q_policy
    blk0, cand, prio = components(name, C["pt"], D["amt_te"].astype(float), C["th"], c_rev, c_blk, qp)
    if not cand.any() or K is None: return blk0, np.zeros_like(blk0)
    kw = {}
    if mode != "offline":
        av = D["amt_va"].astype(float); n_va, n_te = len(D["yva"]), len(D["yte"])
        _, cand_v, prio_v = components(name, C["pv"], av, C["th"], c_rev, c_blk, qp)
        if mode.startswith("online"):
            kw["tau"] = kth_value(prio_v, cand_v, round(K * n_va / n_te))
        else:
            P_te, P_va = len(np.unique(D["per_te"])), len(np.unique(D["per_va"]))
            k_per = max(int(round(K / P_te)), 1)
            kw.update(per=D["per_te"], k_per=k_per, tau0=kth_value(prio_v, cand_v, k_per * P_va))
    rev = select(mode, prio, cand, K, **kw)
    return blk0 & ~rev, rev

def boot_index(t, n=N_BOOT, seed=42):
    rng = np.random.default_rng(seed); m = len(DEC[t]["yte"]); return [rng.integers(0, m, m) for _ in range(n)]

def paired_boot(costs, ref, boots):
    """costs: {policy: per-transaction cost}. Returns one row per competitor of `ref`:
    difference in total cost (competitor - ref) with a 95% percentile interval."""
    names = list(costs); M = np.stack([costs[k] for k in names], 1)
    B = np.array([M[i].sum(0) for i in boots]); j = names.index(ref); out = []
    for k, nm in enumerate(names):
        if nm == ref: continue
        dd = B[:, k] - B[:, j]; lo, hi = np.percentile(dd, [2.5, 97.5])
        out.append(dict(competitor=nm, diff=M[:, k].sum() - M[:, j].sum(), ci_lo=lo, ci_hi=hi, p_worse=(dd <= 0).mean(),
                        verdict="better" if lo > 0 else ("worse" if hi < 0 else "ns")))
    return out

print({t: {k: v.shape for k, v in DEC[t].items()} for t in DEC})
print("periods in test window:", {t: len(np.unique(DEC[t]["per_te"])) for t in DEC})

## Step 10: Calibration and cost-ratio thresholding
ECE (15 bins) and Brier score of raw, Platt, temperature and isotonic
probabilities on the test window; false positives of the cost-ratio threshold
p >= 1/(1+r) with raw, Platt and isotonic probabilities.

In [ ]:
t0 = time.time()
for t in DEC:
    yv, yt = DEC[t]["yva"], DEC[t]["yte"]; y1 = yt == 1; cal_r, cr_r = [], []
    for sc in SCORERS:
        pv, pt = CAL[t][sc]["raw"]["pv"], CAL[t][sc]["raw"]["pt"]
        q = {"raw": pt, "platt": CAL[t][sc]["platt"]["pt"], "temperature": temp(pv, yv, pt), "isotonic": CAL[t][sc]["isotonic"]["pt"]}
        for cn, v in q.items():
            cal_r.append(dict(scorer=sc, calibrator=cn, ECE=ece(yt, v), Brier=brier_score_loss(yt, v)))
        for r in COST_RATIOS:
            for pn in ["raw", "platt", "isotonic"]:
                f = q[pn] >= 1 / (1 + r)
                cr_r.append(dict(scorer=sc, r=r, probs=pn, FP=int((f & ~y1).sum()), FN=int((~f & y1).sum())))
    cal, crt = pd.DataFrame(cal_r), pd.DataFrame(cr_r)
    cal.to_csv(f"{DIRS['output']}/{t}_calibration.csv", index=False); crt.to_csv(f"{DIRS['output']}/{t}_costratio.csv", index=False)
    for _, r_ in cal.iterrows():
        note(f"{t}_ece_{r_.scorer}_{r_.calibrator}", r_.ECE, f"{t}_calibration.csv")
    n_leg = int((~y1).sum()); note(f"{t}_n_legit_test", n_leg, "decision npz")
    for _, r_ in crt[(crt.scorer == "FTT_ens") & (crt.probs == "raw")].iterrows():
        note(f"{t}_costratio_FTT_raw_FP_r{r_.r}", r_.FP, f"{t}_costratio.csv")
    print(f"\n== {DS_NAME[t]}"); print(cal.pivot(index="scorer", columns="calibrator", values="ECE").round(4))
    print(crt[crt.probs != "platt"].pivot_table(index=["scorer", "r"], columns="probs", values="FP").astype(int).T)
print(f"{time.time()-t0:.0f} s")

## Step 11: Offline policy comparison (RQ1, RQ4)
Seven policies x four scorers x budget grid, with isotonic (main) and Platt
probabilities. Paired bootstrap of the budgeted Bayes policy (P7) against each
competitor, and cost sensitivity over C_rev x C_blk at the middle budget.

In [ ]:
t0 = time.time()
for t in DEC:
    D = DEC[t]; y1 = D["yte"] == 1; a = D["amt_te"].astype(float); boots = boot_index(t)
    pol_r, bs_r, sen_r = [], [], []
    for cb in POL_CALIBS:
        for sc in SCORERS:
            for K in K_GRID[t]:
                C = {}
                for p in POLICIES:
                    blk, rev = decide(p, t, sc, cb, K)
                    C[p] = tx_cost(blk, rev, y1, a); cau = (blk | rev) & y1
                    pol_r.append(dict(calib=cb, scorer=sc, K=K, policy=p, n_block=int(blk.sum()), FP_block=int((blk & ~y1).sum()),
                                      TP_block=int((blk & y1).sum()), n_review=int(rev.sum()), TP_review=int((rev & y1).sum()),
                                      recall=cau.sum() / y1.sum(), amount_recall=a[cau].sum() / a[y1].sum(),
                                      cost_block=C_BLK * int((blk & ~y1).sum()), cost_review=C_REV * int(rev.sum()),
                                      cost_missed=a[~blk & ~rev & y1].sum(), cost=C[p].sum()))
                bs_r += [dict(calib=cb, scorer=sc, K=K, **r_) for r_ in paired_boot(C, PROPOSED, boots)]
            for cr in SENS_C_REV:
                for cbk in SENS_C_BLK:
                    cs = {p: tx_cost(*decide(p, t, sc, cb, MID_K[t], c_rev=cr, c_blk=cbk), y1, a, cr, cbk).sum() for p in POLICIES}
                    oth = {k: v for k, v in cs.items() if k != PROPOSED}; bo = min(oth, key=oth.get)
                    sen_r.append(dict(calib=cb, scorer=sc, K=MID_K[t], C_rev=cr, C_blk=cbk, bayes_cost=cs[PROPOSED], best_other=bo,
                                      best_other_cost=oth[bo], rel_gain_pct=100 * (oth[bo] - cs[PROPOSED]) / oth[bo],
                                      bayes_best_or_tied=cs[PROPOSED] <= oth[bo] + 1e-6))
    pol, bs, sen = pd.DataFrame(pol_r), pd.DataFrame(bs_r), pd.DataFrame(sen_r)
    pol.to_csv(f"{DIRS['output']}/{t}_policies.csv", index=False); bs.to_csv(f"{DIRS['output']}/{t}_policy_bootstrap.csv", index=False)
    sen.to_csv(f"{DIRS['output']}/{t}_cost_sensitivity.csv", index=False)
    pi = pol[pol.calib == "isotonic"]; Km = MID_K[t]
    for sc in SCORERS:
        m = pi[(pi.scorer == sc) & (pi.K == Km)].set_index("policy").cost
        for p in POLICIES:
            note(f"{t}_cost_{sc}_K{Km}_{POL_SHORT[p]}", m[p], f"{t}_policies.csv")
        for p in POLICIES[:-1]:
            note(f"{t}_reduction_pct_{sc}_K{Km}_P7_vs_{POL_SHORT[p]}", 100 * (m[p] - m[PROPOSED]) / m[p], f"{t}_policies.csv")
    for cb in POL_CALIBS:
        v = bs[bs.calib == cb].pivot_table(index="competitor", columns="verdict", values="diff", aggfunc="count", fill_value=0)
        for comp in v.index:
            for vd in v.columns: note(f"{t}_boot_{cb}_{POL_SHORT[comp]}_{vd}", int(v.loc[comp, vd]), f"{t}_policy_bootstrap.csv")
        s_ = sen[sen.calib == cb]
        note(f"{t}_sens_{cb}_best_or_tied", int(s_.bayes_best_or_tied.sum()), f"{t}_cost_sensitivity.csv")
        note(f"{t}_sens_{cb}_settings", len(s_), f"{t}_cost_sensitivity.csv")
        note(f"{t}_sens_{cb}_median_gain_pct", float(s_.rel_gain_pct.median()), f"{t}_cost_sensitivity.csv")
    print(f"\n===== {DS_NAME[t]} (isotonic, K = {Km})")
    print(pi[pi.K == Km].pivot_table(index="policy", columns="scorer", values="cost").round(0).astype(int))
    for cb in POL_CALIBS:
        print(f"-- bootstrap verdicts ({cb})")
        print(bs[bs.calib == cb].pivot_table(index="competitor", columns="verdict", values="diff", aggfunc="count", fill_value=0))
        s_ = sen[sen.calib == cb]
        print(f"-- sensitivity ({cb}): best or tied {int(s_.bayes_best_or_tied.sum())}/{len(s_)}, median gain {s_.rel_gain_pct.median():.1f}%")
print(f"{time.time()-t0:.0f} s")

## Step 12: Linear-programming check of Proposition 1
The expected-cost problem with the review budget is solved exactly (HiGHS) at
the middle budget for every dataset and scorer; its optimum is compared with
the expected cost of every policy.

In [ ]:
from scipy.optimize import linprog
from scipy import sparse
t0 = time.time(); opt_r = []
for t in DEC:
    a = DEC[t]["amt_te"].astype(float); Km = MID_K[t]
    for sc in SCORERS:
        p = CAL[t][sc]["isotonic"]["pt"]; n = len(p)
        E = np.stack([p * a, (1 - p) * C_BLK, np.full_like(p, C_REV)], 1)
        A_eq = sparse.kron(sparse.eye(n, format="csr"), np.ones((1, 3)), format="csr")
        A_ub = sparse.csr_matrix((np.ones(n), (np.zeros(n, int), np.arange(n) * 3 + 2)), shape=(1, 3 * n))
        res = linprog(E.ravel(), A_ub=A_ub, b_ub=[Km], A_eq=A_eq, b_eq=np.ones(n), bounds=(0, 1), method="highs")
        x = res.x.reshape(n, 3)
        row = dict(dataset=t, scorer=sc, K=Km, lp_status=res.status, lp_optimum=res.fun, n_fractional=int(((x > 1e-6) & (x < 1 - 1e-6)).sum()))
        for pol in POLICIES:
            b_, r_ = decide(pol, t, sc, "isotonic", Km)
            row[f"E_{pol}"] = (E[:, 0] * (~b_ & ~r_) + E[:, 1] * b_ + E[:, 2] * r_).sum()
        row["gap_P7_minus_lp"] = row[f"E_{PROPOSED}"] - res.fun; opt_r.append(row)
O = pd.DataFrame(opt_r); O.to_csv(f"{DIRS['output']}/optimality_check.csv", index=False)
note("lp_max_abs_gap", float(O.gap_P7_minus_lp.abs().max()), "optimality_check.csv")
note("lp_max_fractional", int(O.n_fractional.max()), "optimality_check.csv")
ec = [c for c in O.columns if c.startswith("E_")]
note("lp_P7_lowest_expected_cost_all_rows", bool((O[ec].idxmin(axis=1) == f"E_{PROPOSED}").all()), "optimality_check.csv")
print(O[["dataset", "scorer", "lp_optimum", f"E_{PROPOSED}", "gap_P7_minus_lp", "n_fractional"]].round(3).to_string(index=False))
print(f"{time.time()-t0:.0f} s")

## Step 13: Window budget enforced on arrival (RQ2)
The five policies with a review step in three modes: `offline` (hindsight
top-K over the test window), `online_uncapped` and `online_capped`
(validation threshold applied in arrival order, without / with the hard cap K).
Paired bootstrap of P7 against the other budgeted policies in both online modes.

In [ ]:
t0 = time.time(); on_r, onb_r = [], []
MODES = ["offline", "online_uncapped", "online_capped"]
for t in DEC:
    D = DEC[t]; y1 = D["yte"] == 1; a = D["amt_te"].astype(float); boots = boot_index(t)
    for sc in SCORERS:
        for K in K_GRID[t]:
            C = {m: {} for m in MODES}
            for p in BUDGETED:
                for m in MODES:
                    blk, rev = decide(p, t, sc, "isotonic", K, mode=m); C[m][p] = tx_cost(blk, rev, y1, a)
                    cau = (blk | rev) & y1
                    on_r.append(dict(dataset=t, scorer=sc, K=K, policy=p, mode=m, n_review=int(rev.sum()), budget_use=rev.sum() / K,
                                     recall=cau.sum() / y1.sum(), amount_recall=a[cau].sum() / a[y1].sum(), cost=C[m][p].sum()))
            for m in ["online_uncapped", "online_capped"]:
                onb_r += [dict(dataset=t, scorer=sc, K=K, mode=m, **r_) for r_ in paired_boot(C[m], PROPOSED, boots)]
R = pd.DataFrame(on_r); RB = pd.DataFrame(onb_r)
R.to_csv(f"{DIRS['output']}/online_policies.csv", index=False); RB.to_csv(f"{DIRS['output']}/online_policy_bootstrap.csv", index=False)
W = R.pivot_table(index=["dataset", "scorer", "K", "policy"], columns="mode", values="cost").reset_index()
W["chg_pct"] = 100 * (W.online_capped - W.offline) / W.offline; W.to_csv(f"{DIRS['output']}/online_vs_offline.csv", index=False)
for t in DEC:
    for p in BUDGETED:
        w = W[(W.dataset == t) & (W.policy == p)].chg_pct
        note(f"{t}_online_chg_median_pct_{POL_SHORT[p]}", float(w.median()), "online_vs_offline.csv")
        note(f"{t}_online_chg_max_pct_{POL_SHORT[p]}", float(w.max()), "online_vs_offline.csv")
        u = R[(R.dataset == t) & (R.policy == p) & (R["mode"] == "online_uncapped")].budget_use
        note(f"{t}_uncapped_budget_use_mean_{POL_SHORT[p]}", float(u.mean()), "online_policies.csv")
        note(f"{t}_uncapped_budget_use_max_{POL_SHORT[p]}", float(u.max()), "online_policies.csv")
    v = RB[(RB.dataset == t) & (RB["mode"] == "online_capped")].pivot_table(index="competitor", columns="verdict", values="diff", aggfunc="count", fill_value=0)
    for comp in v.index:
        for vd in v.columns: note(f"{t}_online_boot_{POL_SHORT[comp]}_{vd}", int(v.loc[comp, vd]), "online_policy_bootstrap.csv")
    print(f"\n== {DS_NAME[t]}: cost change online_capped vs offline (%), median [max]")
    print(W[W.dataset == t].groupby("policy").chg_pct.agg(["median", "max"]).round(1))
    print("-- bootstrap, online_capped"); print(v)
print("\n-- reviews used / K without the cap"); print(R[R["mode"] == "online_uncapped"].groupby(["dataset", "policy"]).budget_use.agg(["mean", "max"]).round(2))
print(f"{time.time()-t0:.0f} s")

## Step 14: Per-period capacity and adaptive threshold (RQ2)
The analyst team is given a capacity per day (IEEE-CIS, BankSim) or per hour
(ULB, whose data span two days): k = round(K / P), with P the number of
periods in the test window. Three modes for every budgeted policy:
`period_hindsight` (best k candidates of each period, an upper bound),
`period_static` (validation threshold, cap k per period) and `period_adaptive`
(the threshold moves after each period halfway towards the value that would
have used that period's capacity, `ADAPT_BETA = 0.5`). Unused capacity and
paired bootstrap of P7 in the adaptive mode are reported.

In [ ]:
t0 = time.time(); pr_r, prb_r = [], []
PMODES = ["period_hindsight", "period_static", "period_adaptive"]
for t in DEC:
    D = DEC[t]; y1 = D["yte"] == 1; a = D["amt_te"].astype(float); boots = boot_index(t); P_te = len(np.unique(D["per_te"]))
    for sc in SCORERS:
        for K in K_GRID[t]:
            k_per = max(int(round(K / P_te)), 1); C = {m: {} for m in PMODES}
            for p in BUDGETED:
                for m in PMODES:
                    blk, rev = decide(p, t, sc, "isotonic", K, mode=m); C[m][p] = tx_cost(blk, rev, y1, a)
                    pr_r.append(dict(dataset=t, scorer=sc, K=K, k_per_period=k_per, periods=P_te, policy=p, mode=m,
                                     n_review=int(rev.sum()), capacity=k_per * P_te, capacity_use=rev.sum() / (k_per * P_te),
                                     cost=C[m][p].sum()))
            for m in ["period_static", "period_adaptive"]:
                prb_r += [dict(dataset=t, scorer=sc, K=K, mode=m, **r_) for r_ in paired_boot(C[m], PROPOSED, boots)]
PR = pd.DataFrame(pr_r); PRB = pd.DataFrame(prb_r)
PR.to_csv(f"{DIRS['output']}/period_policies.csv", index=False); PRB.to_csv(f"{DIRS['output']}/period_policy_bootstrap.csv", index=False)
for t in DEC:
    Km = MID_K[t]; x = PR[(PR.dataset == t) & (PR.K == Km)]
    for sc in ["LightGBM", "FTT_ens"]:
        for p in BUDGETED:
            for m in PMODES:
                r_ = x[(x.scorer == sc) & (x.policy == p) & (x["mode"] == m)].iloc[0]
                note(f"{t}_period_cost_{sc}_K{Km}_{POL_SHORT[p]}_{m}", r_.cost, "period_policies.csv")
                note(f"{t}_period_capuse_{sc}_K{Km}_{POL_SHORT[p]}_{m}", r_.capacity_use, "period_policies.csv")
    v = PRB[(PRB.dataset == t) & (PRB["mode"] == "period_adaptive")].pivot_table(index="competitor", columns="verdict", values="diff", aggfunc="count", fill_value=0)
    for comp in v.index:
        for vd in v.columns: note(f"{t}_period_adaptive_boot_{POL_SHORT[comp]}_{vd}", int(v.loc[comp, vd]), "period_policy_bootstrap.csv")
    print(f"\n== {DS_NAME[t]} (K = {Km}, capacity {x.k_per_period.iloc[0]} per {PERIOD_NAME[t]} over {x.periods.iloc[0]} periods)")
    print(x[x.scorer == "LightGBM"].pivot_table(index="policy", columns="mode", values="cost").round(0).astype(int))
    print(x[x.scorer == "LightGBM"].pivot_table(index="policy", columns="mode", values="capacity_use").round(2))
    print("-- bootstrap, period_adaptive"); print(v)
print(f"{time.time()-t0:.0f} s")

## Step 15: Imperfect analysts
A reviewed fraud is detected with probability q, so a review of a fraud costs
C_rev + (1 - q) * amount in expectation. For q in `Q_GRID` all policies are
evaluated at the middle budget (offline, isotonic). Two versions of the
budgeted Bayes policy are compared: P7 assumes q = 1 when it selects reviews,
P7-q uses the true q in its review gain.

In [ ]:
t0 = time.time(); q_r, qb_r = [], []
for t in DEC:
    D = DEC[t]; y1 = D["yte"] == 1; a = D["amt_te"].astype(float); Km = MID_K[t]; boots = boot_index(t)
    for sc in SCORERS:
        for q in Q_GRID:
            C = {}
            for p in POLICIES:
                blk, rev = decide(p, t, sc, "isotonic", Km, q=q, q_policy=1.0); C[p] = tx_cost(blk, rev, y1, a, q=q)
            blk, rev = decide(PROPOSED, t, sc, "isotonic", Km, q=q, q_policy=q); C["threeway_bayes_budget_q"] = tx_cost(blk, rev, y1, a, q=q)
            n_rev_q = int(rev.sum())
            for p, c in C.items():
                q_r.append(dict(dataset=t, scorer=sc, K=Km, q=q, policy=p, cost=c.sum(), n_review_P7q=n_rev_q))
            qb_r += [dict(dataset=t, scorer=sc, K=Km, q=q, **r_) for r_ in paired_boot(C, "threeway_bayes_budget_q", boots)]
Q = pd.DataFrame(q_r); QB = pd.DataFrame(qb_r)
Q.to_csv(f"{DIRS['output']}/reviewer_accuracy.csv", index=False); QB.to_csv(f"{DIRS['output']}/reviewer_accuracy_bootstrap.csv", index=False)
for t in DEC:
    for sc in ["LightGBM", "FTT_ens"]:
        x = Q[(Q.dataset == t) & (Q.scorer == sc)].pivot_table(index="q", columns="policy", values="cost")
        for q in Q_GRID:
            oth = x.loc[q].drop(["threeway_bayes_budget_q"]); bo = oth.idxmin()
            note(f"{t}_q{q}_{sc}_P7q_vs_best_pct", 100 * (oth[bo] - x.loc[q, "threeway_bayes_budget_q"]) / oth[bo], "reviewer_accuracy.csv")
            note(f"{t}_q{q}_{sc}_P7q_vs_P7_pct", 100 * (x.loc[q, PROPOSED] - x.loc[q, "threeway_bayes_budget_q"]) / x.loc[q, PROPOSED], "reviewer_accuracy.csv")
    print(f"\n== {DS_NAME[t]} (LightGBM, K = {MID_K[t]}): cost by analyst accuracy q")
    print(Q[(Q.dataset == t) & (Q.scorer == "LightGBM")].pivot_table(index="policy", columns="q", values="cost").round(0).astype(int))
print(f"{time.time()-t0:.0f} s")

## Step 16: Cost-scale sweep (RQ3)
C_rev = 10 * lambda and C_blk = 50 * lambda for lambda in `LAMBDAS`; all seven
policies offline at the middle budget with isotonic probabilities. The
advantage of P7 is reported against binary_F1 and against the cheapest
competitor, as a function of the median test fraud amount divided by C_rev.

In [ ]:
t0 = time.time(); sw = []
for t in DEC:
    D = DEC[t]; y1 = D["yte"] == 1; a = D["amt_te"].astype(float); med = float(np.median(a[y1])); note(f"{t}_median_fraud_amount_test", med, "decision npz")
    for sc in SCORERS:
        for lam in LAMBDAS:
            cr, cbk = C_REV * lam, C_BLK * lam
            cs = {p: tx_cost(*decide(p, t, sc, "isotonic", MID_K[t], c_rev=cr, c_blk=cbk), y1, a, cr, cbk).sum() for p in POLICIES}
            oth = {k: v for k, v in cs.items() if k != PROPOSED}; bo = min(oth, key=oth.get)
            sw.append(dict(dataset=t, scorer=sc, K=MID_K[t], lam=lam, C_rev=cr, C_blk=cbk, median_fraud_amount=med,
                           ratio_median_amount_to_C_rev=med / cr, **{f"cost_{k}": v for k, v in cs.items()},
                           adv_vs_F1_pct=100 * (cs["binary_F1"] - cs[PROPOSED]) / cs["binary_F1"],
                           best_other=bo, adv_vs_best_pct=100 * (oth[bo] - cs[PROPOSED]) / oth[bo]))
SW = pd.DataFrame(sw); SW.to_csv(f"{DIRS['output']}/cost_scale_sweep.csv", index=False)
note("costscale_cells", len(SW), "cost_scale_sweep.csv")
note("costscale_cells_positive_vs_best", int((SW.adv_vs_best_pct > 0).sum()), "cost_scale_sweep.csv")
note("costscale_cells_loss_gt2pct", int((SW.adv_vs_best_pct < -2).sum()), "cost_scale_sweep.csv")
note("costscale_min_vs_best_pct", float(SW.adv_vs_best_pct.min()), "cost_scale_sweep.csv")
for t in DEC: note(f"{t}_costscale_median_vs_best_pct", float(SW[SW.dataset == t].adv_vs_best_pct.median()), "cost_scale_sweep.csv")
for c in ["adv_vs_F1_pct", "adv_vs_best_pct"]:
    print(f"\n-- {c}"); print(SW.pivot_table(index=["dataset", "scorer"], columns="lam", values=c).round(1))
print(f"{time.time()-t0:.0f} s")

## Step 17: Calibration sample size (RQ3)
Stratified subsamples of the validation partition with n+ fraud cases (and
the matching number of legitimate cases) are drawn `N_REP` times; isotonic
and Platt calibrators and the F1 threshold are refitted on each subsample and
four policies are evaluated on the full test window at the middle budget.

In [ ]:
t0 = time.time(); CS_POL = ["binary_F1", "binary_bayes_amount", "threeway_expected_loss", PROPOSED]
rng = np.random.default_rng(2026); cs_r = []
for t, grid in CAL_SIZE_GRID.items():
    D = DEC[t]; yv, yt, a = D["yva"], D["yte"], D["amt_te"].astype(float); y1 = yt == 1; Km = MID_K[t]
    pos, neg = np.where(yv == 1)[0], np.where(yv == 0)[0]
    for sc in SCORERS:
        pv, pt = CAL[t][sc]["raw"]["pv"], CAL[t][sc]["raw"]["pt"]
        for npos in grid:
            full = npos >= len(pos)
            for r in range(1 if full else N_REP):
                idx = np.arange(len(yv)) if full else np.concatenate([rng.choice(pos, npos, replace=False),
                       rng.choice(neg, int(round(npos * len(neg) / len(pos))), replace=False)])
                for cb, fn in CAL_FN.items():
                    pc = fn(pv[idx], yv[idx], pt); th = f1_threshold(yv[idx], fn(pv[idx], yv[idx], pv[idx]))
                    costs = {}
                    for p in CS_POL:
                        blk0, cand, prio = components(p, pc, a, th)
                        rev = top_k(prio, cand, Km) if cand.any() else np.zeros_like(cand)
                        costs[p] = tx_cost(blk0 & ~rev, rev, y1, a).sum()
                    cs_r.append(dict(dataset=t, scorer=sc, n_pos=min(npos, len(pos)), rep=r, calib=cb, K=Km, ece_test=ece(yt, pc), **costs))
CS = pd.DataFrame(cs_r)
CS["adv_vs_F1_pct"] = 100 * (CS.binary_F1 - CS[PROPOSED]) / CS.binary_F1
CS["adv_vs_EL_pct"] = 100 * (CS.threeway_expected_loss - CS[PROPOSED]) / CS.threeway_expected_loss
CS["win_vs_F1"] = CS[PROPOSED] < CS.binary_F1; CS.to_csv(f"{DIRS['output']}/calibration_size_runs.csv", index=False)
S = (CS.groupby(["dataset", "scorer", "calib", "n_pos"]).agg(adv_vs_F1_mean=("adv_vs_F1_pct", "mean"),
        adv_vs_F1_q25=("adv_vs_F1_pct", lambda v: v.quantile(.25)), adv_vs_F1_q75=("adv_vs_F1_pct", lambda v: v.quantile(.75)),
        win_rate_vs_F1=("win_vs_F1", "mean"), adv_vs_EL_mean=("adv_vs_EL_pct", "mean"), ece_test_mean=("ece_test", "mean"),
        reps=("rep", "size")).reset_index())
S.to_csv(f"{DIRS['output']}/calibration_size_summary.csv", index=False)
for t in ["ieee", "bs"]:
    s_ = S[(S.dataset == t) & (S.n_pos == 25)]
    note(f"{t}_calsize25_adv_vs_F1_min_pct", float(s_.adv_vs_F1_mean.min()), "calibration_size_summary.csv")
    note(f"{t}_calsize25_adv_vs_F1_max_pct", float(s_.adv_vs_F1_mean.max()), "calibration_size_summary.csv")
    note(f"{t}_calsize25_win_rate_min", float(s_.win_rate_vs_F1.min()), "calibration_size_summary.csv")
for t in CAL_SIZE_GRID:
    print(f"\n-- {DS_NAME[t]}: mean advantage of P7 over binary_F1 (%)")
    print(S[S.dataset == t].pivot_table(index=["scorer", "calib"], columns="n_pos", values="adv_vs_F1_mean").round(1))
print(f"{time.time()-t0:.0f} s")

## Step 18: Paper tables (CSV + LaTeX) and the manuscript number registry
Each table is written as CSV and as a booktabs LaTeX fragment
(`output/tables/`). `output/manuscript_numbers.csv` lists every number quoted
in the manuscript with the file it comes from.

In [ ]:
TB = f"{DIRS['output']}/tables"
def to_tex(df, name, caption, label, colfmt=None, note_=None):
    df.to_csv(f"{TB}/{name}.csv", index=False)
    cols = list(df.columns); colfmt = colfmt or ("l" * 1 + "r" * (len(cols) - 1))
    esc = lambda s: str(s).replace("%", "\\%").replace("_", "\\_").replace("&", "\\&")
    L = ["\\begin{table}[t]", "\\centering", f"\\caption{{{caption}}}", f"\\label{{{label}}}",
         f"\\begin{{tabular}}{{{colfmt}}}", "\\toprule", " & ".join(esc(c) for c in cols) + " \\\\", "\\midrule"]
    L += [" & ".join(esc(v) for v in row) + " \\\\" for row in df.astype(str).values.tolist()]
    L += ["\\bottomrule", "\\end{tabular}"]
    if note_: L.append(f"\\par\\smallskip{{\\footnotesize {note_}}}")
    L.append("\\end{table}"); open(f"{TB}/{name}.tex", "w").write("\n".join(L) + "\n")
fmt = lambda v, d=0: f"{v:,.{d}f}"

# Table 1: datasets
src = {"ieee": "ieee_tree.npz", "ulb": "ulb.npz", "bs": "banksim.npz"}; rows = []
for t in DEC:
    Z = np.load(f"{DIRS['dataset']}/{src[t]}"); D = DEC[t]
    for part, y_, am in [("Train", Z["ytr"], None), ("Validation", D["yva"], D["amt_va"]), ("Test", D["yte"], D["amt_te"])]:
        rows.append({"Dataset": DS_NAME[t] if part == "Train" else "", "Partition": part, "Transactions": fmt(len(y_)),
                     "Frauds": fmt(int(y_.sum())), "Fraud rate (%)": fmt(100 * y_.mean(), 2),
                     "Mean amount, fraud": "--" if am is None else fmt(am[y_ == 1].mean(), 2),
                     "Mean amount, legitimate": "--" if am is None else fmt(am[y_ == 0].mean(), 2)})
        note(f"{t}_{part.lower()}_n", len(y_), src[t]); note(f"{t}_{part.lower()}_fraud", int(y_.sum()), src[t])
to_tex(pd.DataFrame(rows), "table1_datasets", "Datasets after the chronological 70/15/15 split. Amounts are in the units of each dataset.", "tab:data", "llrrrrr")

# Table 2: detection performance
rows = []
for t in DEC:
    ph = pd.read_csv(f"{DIRS['output']}/{t}_phase1_results.csv").set_index("model")
    for m in ph.index:
        note(f"{t}_AUPRC_{m}", float(ph.loc[m, "AUPRC"]), f"{t}_phase1_results.csv"); note(f"{t}_AUROC_{m}", float(ph.loc[m, "AUROC"]), f"{t}_phase1_results.csv")
models = ["LogReg", "MLP", "RandomForest", "XGBoost", "LightGBM", "CatBoost", "FTT_ens"]
mname = {"LogReg": "Logistic regression", "MLP": "MLP", "RandomForest": "Random forest", "FTT_ens": "FT-Transformer (ensemble)"}
for m in models:
    r_ = {"Model": mname.get(m, m)}
    for t in DEC:
        ph = pd.read_csv(f"{DIRS['output']}/{t}_phase1_results.csv").set_index("model")
        r_[f"{DS_NAME[t]} AUPRC"] = fmt(ph.loc[m, "AUPRC"], 3); r_[f"{DS_NAME[t]} AUROC"] = fmt(ph.loc[m, "AUROC"], 3)
    rows.append(r_)
L1 = pd.read_csv(f"{DIRS['output']}/ieee_ftt_training_log.csv"); L2 = pd.read_csv(f"{DIRS['output']}/ulb_bs_ftt_training_log.csv")
seed_auprc = {"ieee": L1.test_auprc, "ulb": L2[(L2.tag == "ulb") & (L2.name == "FTT")].test_auprc, "bs": L2[(L2.tag == "bs") & (L2.name == "FTT")].test_auprc}
r_ = {"Model": "FT-Transformer (single seed, mean $\\pm$ sd)"}
for t in DEC:
    v = seed_auprc[t]; r_[f"{DS_NAME[t]} AUPRC"] = f"{v.mean():.3f} $\\pm$ {v.std(ddof=1):.3f}"; r_[f"{DS_NAME[t]} AUROC"] = "--"
    note(f"{t}_FTT_seed_auprc_mean", float(v.mean()), "ftt training logs"); note(f"{t}_FTT_seed_auprc_sd", float(v.std(ddof=1)), "ftt training logs")
rows.append(r_)
note("ieee_FTT_seeds_at_epoch_cap", int(L1.hit_ceiling.sum()), "ieee_ftt_training_log.csv")
for nm in ["ABL-bceL2", "ABL-focalL1"]:
    v = L2[(L2.tag == "ulb") & (L2.name == nm)].test_auprc
    note(f"ulb_{nm}_auprc_mean", float(v.mean()), "ulb_bs_ftt_training_log.csv"); note(f"ulb_{nm}_auprc_sd", float(v.std(ddof=1)), "ulb_bs_ftt_training_log.csv")
to_tex(pd.DataFrame(rows), "table2_detection", "Detection performance on the test partitions.", "tab:detection", "lrrrrrr")

# Table 3: calibration (ECE)
rows = []
for t in DEC:
    c_ = pd.read_csv(f"{DIRS['output']}/{t}_calibration.csv").pivot(index="scorer", columns="calibrator", values="ECE")
    for sc in SCORERS:
        rows.append({"Dataset": DS_NAME[t] if sc == SCORERS[0] else "", "Scorer": SC_NAME[sc],
                     **{k: fmt(c_.loc[sc, k], 4) for k in ["raw", "platt", "temperature", "isotonic"]}})
to_tex(pd.DataFrame(rows).rename(columns={"raw": "Raw", "platt": "Platt", "temperature": "Temperature", "isotonic": "Isotonic"}),
       "table3_calibration", "Expected calibration error (15 bins) on the test partitions.", "tab:ece", "llrrrr")

# Table 4: policy costs at the middle budget (thousands), LightGBM and FT-Transformer
rows = []
for p in POLICIES:
    r_ = {"Policy": POL_NAME[p]}
    for t in DEC:
        P = pd.read_csv(f"{DIRS['output']}/{t}_policies.csv"); P = P[(P.calib == "isotonic") & (P.K == MID_K[t])]
        for sc in ["LightGBM", "FTT_ens"]:
            r_[f"{DS_NAME[t]} {SC_NAME[sc]}"] = fmt(P[(P.scorer == sc) & (P.policy == p)].cost.iloc[0] / 1e3, 2 if t == "ulb" else 1)
    rows.append(r_)
to_tex(pd.DataFrame(rows), "table4_policy_costs", "Total cost on the test window (thousands) at the middle budget (IEEE-CIS $K=2000$, ULB $K=100$, BankSim $K=400$), isotonic calibration, $C_{\\mathrm{rev}}=10$, $C_{\\mathrm{blk}}=50$.", "tab:costs", "lrrrrrr")

# Table 5: bootstrap verdicts of P7 (better / not significant / worse)
B_off = pd.concat([pd.read_csv(f"{DIRS['output']}/{t}_policy_bootstrap.csv").assign(dataset=t) for t in DEC])
B_off = B_off[B_off.calib == "isotonic"].assign(mode="offline")
B_on = pd.read_csv(f"{DIRS['output']}/online_policy_bootstrap.csv"); B_on = B_on[B_on["mode"] == "online_capped"]
B_pa = pd.read_csv(f"{DIRS['output']}/period_policy_bootstrap.csv"); B_pa = B_pa[B_pa["mode"] == "period_adaptive"]
rows = []
for p in POLICIES[:-1]:
    r_ = {"Competitor": POL_NAME[p]}
    for t in DEC:
        for lab, B in [("offline", B_off), ("online", B_on), ("per-period", B_pa)]:
            b = B[(B.dataset == t) & (B.competitor == p)]
            r_[f"{DS_NAME[t]} {lab}"] = "--" if b.empty else "/".join(str(int((b.verdict == v).sum())) for v in ["better", "ns", "worse"])
    rows.append(r_)
to_tex(pd.DataFrame(rows), "table5_bootstrap", "Paired bootstrap of the budgeted Bayes policy (P7) against each competitor: number of cells (scorer $\\times$ budget) in which P7 is significantly cheaper / not significantly different / significantly more expensive (95\\% interval, 1,000 resamples). Offline: hindsight selection; online: window budget enforced on arrival with a hard cap; per-period: per-period capacity with the adaptive threshold.", "tab:boot", "l" + "r" * 9)

# Table 6: operational modes at the middle budget, LightGBM
R = pd.read_csv(f"{DIRS['output']}/online_policies.csv"); PR = pd.read_csv(f"{DIRS['output']}/period_policies.csv"); rows = []
for t in DEC:
    for p in BUDGETED:
        g = lambda df, m, col="cost": df[(df.dataset == t) & (df.scorer == "LightGBM") & (df.K == MID_K[t]) & (df.policy == p) & (df["mode"] == m)][col].iloc[0]
        off, onc = g(R, "offline"), g(R, "online_capped")
        rows.append({"Dataset": DS_NAME[t] if p == BUDGETED[0] else "", "Policy": POL_SHORT[p],
                     "Offline": fmt(off, 0), "Online capped": fmt(onc, 0), "Change (%)": fmt(100 * (onc - off) / off, 1),
                     "Reviews/K, uncapped": fmt(g(R, "online_uncapped", "budget_use"), 2),
                     "Per-period static": fmt(g(PR, "period_static"), 0), "Per-period adaptive": fmt(g(PR, "period_adaptive"), 0),
                     "Capacity use, adaptive": fmt(g(PR, "period_adaptive", "capacity_use"), 2)})
to_tex(pd.DataFrame(rows), "table6_operational", "Budget enforcement in operation (LightGBM, middle budget): total cost with hindsight selection, with the window budget enforced on arrival, and with a per-period capacity; review volume relative to $K$ without the cap; share of per-period capacity used by the adaptive threshold.", "tab:operational", "llrrrrrrr")

# Table 7: calibration sample size (LightGBM and FT-Transformer, isotonic and Platt)
S = pd.read_csv(f"{DIRS['output']}/calibration_size_summary.csv"); rows = []
for t in ["ieee", "bs", "ulb"]:
    for sc in ["LightGBM", "FTT_ens"]:
        for cb in POL_CALIBS:
            s_ = S[(S.dataset == t) & (S.scorer == sc) & (S.calib == cb)].set_index("n_pos")
            r_ = {"Dataset": DS_NAME[t], "Scorer": SC_NAME[sc], "Calibrator": cb}
            for n_ in s_.index: r_[f"n+={n_}"] = fmt(s_.loc[n_, "adv_vs_F1_mean"], 1)
            rows.append(r_)
T7 = pd.DataFrame(rows).fillna("--"); T7.to_csv(f"{TB}/table7_calibration_size.csv", index=False)
print("tables:", sorted(os.listdir(TB)))

pd.DataFrame(NUMBERS).to_csv(f"{DIRS['output']}/manuscript_numbers.csv", index=False)
print(f"{len(NUMBERS)} numbers registered in manuscript_numbers.csv")

## Step 19: Main figures (Figure_1 to Figure_9)
Decision Support Systems artwork: full text width (7.0 in), 8 pt text, no
title inside the figure (the caption carries it), panel labels centred below
the x-axis. Each figure is saved as vector PDF, 600 dpi PNG and a CSV with the
plotted data. Figure 1 is also written as an editable draw.io file.

In [ ]:
import html
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
plt.rcParams.update({"font.size": 8, "axes.labelsize": 8, "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7,
                     "axes.grid": True, "grid.alpha": .3, "grid.linewidth": .5, "axes.linewidth": .6, "lines.linewidth": 1.1,
                     "lines.markersize": 3.5, "pdf.fonttype": 42, "axes.axisbelow": True})
FIG = DIRS["fig"]; TAGS = ["ieee", "ulb", "bs"]
PS = {"binary_F1": ("#7f7f7f", "s", "--"), "binary_bayes_amount": ("#8c564b", "D", "--"), "topK_review": ("#bcbd22", "v", ":"),
      "reject_option": ("#9467bd", "^", ":"), "threeway_fixed": ("#1f77b4", "o", "-."), "threeway_expected_loss": ("#2ca02c", "P", "-."),
      "threeway_bayes_budget": ("#d62728", "o", "-"), "threeway_bayes_budget_q": ("#000000", "*", "-")}
def panel(ax, s, dy=-0.18):
    ax.text(0.5, dy, s, transform=ax.transAxes, ha="center", va="top", fontsize=8, fontweight="bold")
def save(fig, name, data):
    fig.savefig(f"{FIG}/{name}.pdf", bbox_inches="tight"); fig.savefig(f"{FIG}/{name}.png", dpi=600, bbox_inches="tight")
    (data if isinstance(data, pd.DataFrame) else pd.DataFrame(data)).to_csv(f"{FIG}/{name}.csv", index=False); plt.close(fig); print("saved", name)
def logk_axis(ax, kg):
    ax.set_xscale("log"); ax.set_xticks(kg); ax.set_xticklabels([f"{v:,}" for v in kg], fontsize=6); ax.minorticks_off(); ax.set_xlabel("Review budget $K$")

# ---------- Figure 1: decision framework (canvas units: 1 = 0.01 in)
CW, CH = 700, 400
BX = {"in":  (10, 20, 150, 110, "Input", "Transaction features $\\mathbf{x}_i$\nand amount $a_i$", "#dae8fc", "#6c8ebf"),
      "sc":  (200, 20, 210, 110, "Stage 1: Scoring model", "any classifier\n(XGBoost, LightGBM,\nCatBoost, FT-Transformer)", "#e1d5e7", "#9673a6"),
      "cal": (450, 20, 240, 110, "Stage 2: Calibration", "isotonic (main) or Platt\nfitted on validation data\n$p_i = g(s(\\mathbf{x}_i))$", "#ffe6cc", "#d79b00"),
      "ec":  (10, 175, 250, 120, "Stage 3: Expected cost", "approve: $p_i a_i$\nblock: $(1-p_i)\\,C_{\\mathrm{blk}}$\nreview: $C_{\\mathrm{rev}}$", "#fff2cc", "#d6b656"),
      "sel": (300, 175, 270, 120, "Stage 4: Budgeted selection", "gain $\\Delta_i = \\min\\{p_i a_i, (1-p_i)C_{\\mathrm{blk}}\\} - C_{\\mathrm{rev}}$\nreview if $\\Delta_i>0$ and among the\n$K$ largest (offline) or $\\Delta_i\\geq\\tau$ (online);\notherwise the cheaper automatic action", "#f8cecc", "#b85450")}
OUTS = [("APPROVE", "#d5e8d4", "#82b366"), ("REVIEW ($\\leq$ K)", "#fff2cc", "#d6b656"), ("BLOCK", "#f8cecc", "#b85450")]; oy = [175, 220, 265]
fig, ax = plt.subplots(figsize=(CW / 100, CH / 100)); ax.set_xlim(0, CW); ax.set_ylim(CH, 0); ax.axis("off")
def box(x, y, w, h, fc, ec, lw=1.1):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0,rounding_size=6", fc=fc, ec=ec, lw=lw))
for k, (x, y, w, h, t_, b, fc, ec) in BX.items():
    box(x, y, w, h, fc, ec); ax.text(x + w / 2, y + 16, t_, ha="center", va="center", fontsize=8, fontweight="bold")
    ax.text(x + w / 2, y + (h + 24) / 2, b, ha="center", va="center", fontsize=7.2, linespacing=1.35)
for (t_, fc, ec), y in zip(OUTS, oy):
    box(588, y, 107, 30, fc, ec); ax.text(641.5, y + 15, t_, ha="center", va="center", fontsize=6.8, fontweight="bold")
def arr(p, q, txt=None):
    ax.annotate("", q, p, arrowprops=dict(arrowstyle="-|>", lw=1.0, color="k", shrinkA=0, shrinkB=0))
    if txt: ax.text((p[0] + q[0]) / 2, (p[1] + q[1]) / 2 - 6, txt, ha="center", va="bottom", fontsize=6.8, style="italic")
arr((160, 75), (200, 75)); arr((410, 75), (450, 75), "$s(\\mathbf{x}_i)$")
ax.plot([570, 570, 135], [130, 152, 152], "k-", lw=1.0); arr((135, 152), (135, 175)); ax.text(360, 148, "$p_i$", ha="center", va="bottom", fontsize=6.8, style="italic")
arr((260, 235), (300, 235))
for y in oy: arr((570, 235), (588, y + 15))
box(5, 315, 690, 75, "#f5f5f5", "#666666", lw=.8)
ax.text(350, 352, "Protocol: chronological 70/15/15 split; models trained on the training window and early-stopped on validation;\n"
        "calibrator, F1 threshold and online review threshold $\\tau$ fitted on validation only; the test window is used once.\n"
        "Evaluation: realised cost $C_{\\mathrm{blk}}\\times$false blocks $+\\,C_{\\mathrm{rev}}\\times$reviews $+$ amount of approved fraud.",
        ha="center", va="center", fontsize=6.5, linespacing=1.4)
save(fig, "Figure_1", [dict(element=k, x=v[0], y=v[1], w=v[2], h=v[3], title=v[4]) for k, v in BX.items()])
cells = ['<mxCell id="0"/>', '<mxCell id="1" parent="0"/>']
plain = {"in": "Input|Transaction features x_i and amount a_i", "sc": "Stage 1: Scoring model|any classifier (XGBoost, LightGBM, CatBoost, FT-Transformer)",
         "cal": "Stage 2: Calibration|isotonic (main) or Platt, fitted on validation data; p_i = g(s(x_i))",
         "ec": "Stage 3: Expected cost|approve: p_i a_i; block: (1-p_i) C_blk; review: C_rev",
         "sel": "Stage 4: Budgeted selection|gain = min(p_i a_i, (1-p_i) C_blk) - C_rev; review if gain > 0 and among the K largest (offline) or gain >= tau (online); otherwise the cheaper automatic action"}
def v_(id_, val, x, y, w, h, fc, ec, extra=""):
    cells.append(f'<mxCell id="{id_}" value="{html.escape(val)}" style="rounded=1;whiteSpace=wrap;html=1;fillColor={fc};strokeColor={ec};fontSize=11;arcSize=6;{extra}" vertex="1" parent="1"><mxGeometry x="{x*2}" y="{y*2}" width="{w*2}" height="{h*2}" as="geometry"/></mxCell>')
for k, (x, y, w, h, t_, b, fc, ec) in BX.items():
    tt, bb = plain[k].split("|"); v_(k, f"<b>{tt}</b><br>{bb}", x, y, w, h, fc, ec)
for i, ((t_, fc, ec), y) in enumerate(zip(OUTS, oy)): v_(f"o{i}", ["APPROVE", "REVIEW (&lt;= K)", "BLOCK"][i], 588, y, 107, 30, fc, ec, "fontStyle=1;")
v_("prot", "Protocol: chronological 70/15/15 split; models trained on the training window and early-stopped on validation; calibrator, F1 threshold and online review threshold tau fitted on validation only; the test window is used once. Evaluation: realised cost = C_blk x false blocks + C_rev x reviews + amount of approved fraud.", 5, 315, 690, 75, "#f5f5f5", "#666666")
for i, (a_, b_) in enumerate([("in", "sc"), ("sc", "cal"), ("cal", "ec"), ("ec", "sel"), ("sel", "o0"), ("sel", "o1"), ("sel", "o2")]):
    cells.append(f'<mxCell id="e{i}" style="edgeStyle=orthogonalEdgeStyle;html=1;endArrow=block;endFill=1;" edge="1" parent="1" source="{a_}" target="{b_}"><mxGeometry relative="1" as="geometry"/></mxCell>')
open(f"{FIG}/Figure_1.drawio", "w").write('<mxfile host="app.diagrams.net"><diagram name="framework"><mxGraphModel grid="1" gridSize="10" page="1" pageWidth="1400" pageHeight="800"><root>' + "".join(cells) + '</root></mxGraphModel></diagram></mxfile>')

# ---------- Figure 2: decision regions of P7 (LightGBM, isotonic, middle budget)
fig, axes = plt.subplots(1, 3, figsize=(FIG_W, 2.7)); pts, summ = [], []; rng_ = np.random.default_rng(0)
for k, t in enumerate(TAGS):
    D = DEC[t]; yt, a = D["yte"], D["amt_te"].astype(float); p = CAL[t]["LightGBM"]["isotonic"]["pt"]; K = MID_K[t]
    blk, rev = decide(PROPOSED, t, "LightGBM", "isotonic", K)
    gain = np.minimum(p * a, (1 - p) * C_BLK) - C_REV; gpos = np.sort(gain[gain > 0])[::-1]
    gk = gpos[K - 1] if len(gpos) >= K else 0.0; thr = C_REV + max(gk, 0.0); pmax = 1 - thr / C_BLK
    pp, aa = np.maximum(p, 1e-4), np.maximum(a, 1e-2); dec = np.where(blk, "block", np.where(rev, "review", "approve"))
    keep = (yt == 1) | (rng_.random(len(yt)) < min(1, 20000 / max((yt == 0).sum(), 1))); ax = axes[k]
    for d_, c in [("approve", "#bdbdbd"), ("review", "#ff7f0e"), ("block", "#d62728")]:
        m0 = keep & (dec == d_) & (yt == 0); m1 = (dec == d_) & (yt == 1)
        ax.scatter(pp[m0], aa[m0], s=1, c=c, alpha=.35, lw=0, rasterized=True); ax.scatter(pp[m1], aa[m1], s=7, c=c, marker="x", lw=.6, rasterized=True)
    g = np.logspace(-4, 0, 400)[:-1]; ax.plot(g, (1 - g) * C_BLK / g, "k-", lw=.9)
    ok = g <= pmax; ax.plot(g[ok], thr / g[ok], "k--", lw=.9)
    if pmax > 0: ax.plot([pmax, pmax], [thr / pmax, aa.max() * 1.3], "k--", lw=.9)
    ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlim(8e-5, 1.1); ax.set_ylim(aa.min() * .8, aa.max() * 1.3)
    ax.set_xlabel("Calibrated fraud probability $p$"); ax.set_ylabel("Transaction amount $a$" if k == 0 else "")
    panel(ax, f"({'abc'[k]}) {DS_NAME[t]}, $K$ = {K:,}", -0.25)
    sel = np.where(keep)[0]; pts.append(pd.DataFrame({"dataset": t, "p": p[sel], "amount": a[sel], "y": yt[sel], "decision": dec[sel]}))
    summ.append(dict(dataset=t, K=K, gain_threshold=gk, review_p_max=pmax, fraud_blocked=int((blk & (yt == 1)).sum()),
                     fraud_reviewed=int((rev & (yt == 1)).sum()), fraud_approved=int((~blk & ~rev & (yt == 1)).sum())))
h_ = [mpatches.Patch(color=c, label=l) for l, c in [("approve", "#bdbdbd"), ("review", "#ff7f0e"), ("block", "#d62728")]]
h_ += [plt.Line2D([], [], color="k", marker="x", ls="", ms=4, label="fraud"), plt.Line2D([], [], color="k", ls="-", label="approve/block boundary"),
       plt.Line2D([], [], color="k", ls="--", label="review region boundary")]
fig.legend(handles=h_, loc="upper center", ncol=6, bbox_to_anchor=(0.5, 1.07), frameon=False); fig.tight_layout()
save(fig, "Figure_2", pd.concat(pts)); pd.DataFrame(summ).to_csv(f"{FIG}/Figure_2_summary.csv", index=False)

# ---------- Figure 3: total cost versus budget, LightGBM and FT-Transformer
POLD = {t: pd.read_csv(f"{DIRS['output']}/{t}_policies.csv") for t in TAGS}
fig, axes = plt.subplots(2, 3, figsize=(FIG_W, 5.0)); rows = []
for i, sc in enumerate(["LightGBM", "FTT_ens"]):
    for k, t in enumerate(TAGS):
        ax = axes[i, k]; P = POLD[t][(POLD[t].calib == "isotonic") & (POLD[t].scorer == sc)]
        for p in POLICIES:
            s = P[P.policy == p].sort_values("K"); c, m, ls = PS[p]
            ax.plot(s.K, s.cost / 1e3, ls=ls, marker=m, color=c, lw=1.8 if p == PROPOSED else .9, zorder=5 if p == PROPOSED else 2, label=POL_NAME[p])
            rows += [dict(scorer=sc, dataset=t, policy=p, K=a_, cost=b_) for a_, b_ in zip(s.K, s.cost)]
        logk_axis(ax, K_GRID[t])
        if t == "bs":
            ymax = 1.15 * P[P.policy.isin(["binary_F1", "binary_bayes_amount", "threeway_fixed"])].cost.max() / 1e3; ax.set_ylim(0, ymax)
        if k == 0: ax.set_ylabel(f"Total cost, thousands\n({SC_NAME[sc]})")
        panel(ax, f"({'abcdef'[3*i+k]}) {DS_NAME[t]}, {SC_NAME[sc]}", -0.27)
h_, l_ = axes[0, 0].get_legend_handles_labels(); fig.legend(h_, l_, loc="upper center", ncol=4, bbox_to_anchor=(0.5, 1.06), frameon=False)
fig.tight_layout(h_pad=2.2); save(fig, "Figure_3", rows)

# ---------- Figure 4: cost decomposition at the middle budget, LightGBM
fig, axes = plt.subplots(1, 3, figsize=(FIG_W, 2.9)); rows = []
for k, t in enumerate(TAGS):
    ax = axes[k]; P = POLD[t]; P = P[(P.calib == "isotonic") & (P.scorer == "LightGBM") & (P.K == MID_K[t])].set_index("policy").loc[POLICIES]
    x = np.arange(len(POLICIES)); bot = np.zeros(len(x))
    for comp, c, nm in [("cost_missed", "#d62728", "Approved fraud (amount)"), ("cost_block", "#1f77b4", "False blocks"), ("cost_review", "#ff7f0e", "Reviews")]:
        ax.bar(x, P[comp] / 1e3, bottom=bot, color=c, edgecolor="k", lw=.3, width=.75, label=nm); bot += P[comp].values / 1e3
    ax.set_xticks(x); ax.set_xticklabels([POL_SHORT[p] for p in POLICIES]); ax.set_ylabel("Total cost (thousands)" if k == 0 else "")
    top = 1.25 * P.drop("topK_review").cost.max() / 1e3
    if P.loc["topK_review", "cost"] / 1e3 > top:
        ax.set_ylim(0, top); ax.annotate(f"{P.loc['topK_review','cost']/1e3:.0f}", (2, top * .99), ha="center", va="top", fontsize=6)
    panel(ax, f"({'abc'[k]}) {DS_NAME[t]}, $K$ = {MID_K[t]:,}", -0.16)
    rows.append(P[["cost_missed", "cost_block", "cost_review", "cost"]].reset_index().assign(dataset=t))
h_, l_ = axes[0].get_legend_handles_labels(); fig.legend(h_, l_, loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.07), frameon=False)
fig.tight_layout(); save(fig, "Figure_4", pd.concat(rows))

# ---------- Figure 5: paired-bootstrap intervals at the middle budget
comp = POLICIES[:-1]; fig, axes = plt.subplots(1, 3, figsize=(FIG_W, 2.7), sharey=True); rows = []
for k, t in enumerate(TAGS):
    ax = axes[k]; B = pd.read_csv(f"{DIRS['output']}/{t}_policy_bootstrap.csv"); B = B[(B.calib == "isotonic") & (B.K == MID_K[t])]
    for sc, c, off, mk in [("LightGBM", "#2ca02c", -.17, "o"), ("FTT_ens", "#d62728", .17, "s")]:
        b = B[B.scorer == sc].set_index("competitor").loc[comp]; y = np.arange(len(comp)) + off
        ax.errorbar(b["diff"] / 1e3, y, xerr=[(b["diff"] - b.ci_lo) / 1e3, (b.ci_hi - b["diff"]) / 1e3], fmt=mk, color=c, ms=3, capsize=1.5, lw=.8, label=SC_NAME[sc])
        rows.append(b.reset_index().assign(dataset=t, scorer=sc))
    ax.axvline(0, color="k", lw=.8)
    lo, hi = B.ci_lo.min() / 1e3, B.ci_hi.max() / 1e3
    if hi > 20:  # wide range: symmetric log axis with plain tick labels
        tk = [v for v in [-100, -10, -1, 0, 1, 10, 100, 1000] if lo * 1.5 <= v <= hi * 1.5 or v == 0]
        ax.set_xscale("symlog", linthresh=1, linscale=.6); ax.set_xticks(tk); ax.set_xticklabels([f"{v:g}" for v in tk]); ax.minorticks_off()
        ax.set_xlim(min(lo * 1.5, -0.5) if lo < 0 else -0.5, hi * 1.5)
    ax.set_yticks(range(len(comp))); ax.set_yticklabels([POL_NAME[p] for p in comp]); ax.set_xlabel("Cost difference (thousands)")
    panel(ax, f"({'abc'[k]}) {DS_NAME[t]}, $K$ = {MID_K[t]:,}", -0.25)
h_, l_ = axes[0].get_legend_handles_labels(); fig.legend(h_, l_, loc="upper center", ncol=2, bbox_to_anchor=(0.6, 1.06), frameon=False)
fig.tight_layout(); save(fig, "Figure_5", pd.concat(rows))

# ---------- Figure 6: window budget enforced on arrival (LightGBM)
R = pd.read_csv(f"{DIRS['output']}/online_policies.csv"); fig, axes = plt.subplots(2, 3, figsize=(FIG_W, 4.8)); rows = []
for k, t in enumerate(TAGS):
    r = R[(R.dataset == t) & (R.scorer == "LightGBM")]; kg = K_GRID[t]
    for p in BUDGETED:
        c, m, ls = PS[p]; lw = 1.8 if p == PROPOSED else .9
        on = r[(r.policy == p) & (r["mode"] == "online_capped")].sort_values("K"); off = r[(r.policy == p) & (r["mode"] == "offline")].sort_values("K")
        axes[0, k].plot(on.K, on.cost / 1e3, ls=ls, marker=m, color=c, lw=lw, label=POL_NAME[p])
        axes[1, k].plot(kg, 100 * (on.cost.values - off.cost.values) / off.cost.values, ls=ls, marker=m, color=c, lw=lw)
        rows += [dict(dataset=t, policy=p, K=a_, offline=b_, online_capped=d_) for a_, b_, d_ in zip(kg, off.cost, on.cost)]
    y0 = r[(r["mode"] == "online_capped") & (r.policy != "topK_review")].cost.max() / 1e3 * 1.15; axes[0, k].set_ylim(0, y0)
    for i in (0, 1): logk_axis(axes[i, k], kg)
    axes[1, k].axhline(0, color="k", lw=.8)
    axes[0, k].set_ylabel("Total cost, online capped\n(thousands)" if k == 0 else ""); axes[1, k].set_ylabel("Change vs offline (%)" if k == 0 else "")
    panel(axes[0, k], f"({'abc'[k]}) {DS_NAME[t]}", -0.27); panel(axes[1, k], f"({'def'[k]}) {DS_NAME[t]}", -0.27)
h_, l_ = axes[0, 0].get_legend_handles_labels(); fig.legend(h_, l_, loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.07), frameon=False)
fig.tight_layout(h_pad=2.2); save(fig, "Figure_6", rows)

# ---------- Figure 7: per-period capacity (top) and analyst accuracy (bottom), LightGBM
PR = pd.read_csv(f"{DIRS['output']}/period_policies.csv"); Q = pd.read_csv(f"{DIRS['output']}/reviewer_accuracy.csv")
fig, axes = plt.subplots(2, 3, figsize=(FIG_W, 4.9)); rows = []
for k, t in enumerate(TAGS):
    ax = axes[0, k]; r = PR[(PR.dataset == t) & (PR.scorer == "LightGBM")]
    for p in BUDGETED:
        c, m, ls = PS[p]; s = r[(r.policy == p) & (r["mode"] == "period_adaptive")].sort_values("K")
        ax.plot(s.K, s.cost / 1e3, ls=ls, marker=m, color=c, lw=1.8 if p == PROPOSED else .9, label=POL_NAME[p])
        rows += [dict(part="period_adaptive", dataset=t, policy=p, K=a_, cost=b_) for a_, b_ in zip(s.K, s.cost)]
    s = r[(r.policy == PROPOSED) & (r["mode"] == "period_hindsight")].sort_values("K")
    ax.plot(s.K, s.cost / 1e3, color="#d62728", ls=(0, (1, 1)), lw=1, label="Budgeted Bayes (P7), per-period hindsight")
    ax.set_ylim(0, r[(r["mode"] == "period_adaptive") & (r.policy != "topK_review")].cost.max() / 1e3 * 1.15)
    logk_axis(ax, K_GRID[t]); ax.set_ylabel("Total cost, per-period capacity\n(thousands)" if k == 0 else "")
    panel(ax, f"({'abc'[k]}) {DS_NAME[t]}, capacity per {PERIOD_NAME[t]}", -0.27)
    ax = axes[1, k]; x = Q[(Q.dataset == t) & (Q.scorer == "LightGBM")]
    for p in ["binary_F1", "binary_bayes_amount", "threeway_expected_loss", PROPOSED, "threeway_bayes_budget_q"]:
        c, m, ls = PS[p]; s = x[x.policy == p].sort_values("q")
        ax.plot(s.q, s.cost / 1e3, ls=ls, marker=m, color=c, lw=1.6 if "budget" in p else .9,
                label="Budgeted Bayes with $q$ (P7-$q$)" if p.endswith("_q") else POL_NAME[p])
        rows += [dict(part="analyst_accuracy", dataset=t, policy=p, q=a_, cost=b_) for a_, b_ in zip(s.q, s.cost)]
    ax.set_xlabel("Analyst detection probability $q$"); ax.set_ylabel("Total cost, $K$ at middle value\n(thousands)" if k == 0 else "")
    panel(ax, f"({'def'[k]}) {DS_NAME[t]}, $K$ = {MID_K[t]:,}", -0.27)
h1, l1 = axes[0, 0].get_legend_handles_labels(); h2, l2 = axes[1, 0].get_legend_handles_labels()
H, Lb = [], []
for h__, l__ in zip(h1 + h2, l1 + l2):
    if l__ not in Lb: H.append(h__); Lb.append(l__)
fig.legend(H, Lb, loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.09), frameon=False)
fig.tight_layout(h_pad=2.2); save(fig, "Figure_7", rows)

# ---------- Figure 8: cost-scale sweep
SW = pd.read_csv(f"{DIRS['output']}/cost_scale_sweep.csv"); CL = {"ieee": "#1f77b4", "ulb": "#d62728", "bs": "#2ca02c"}
fig, axes = plt.subplots(1, 2, figsize=(FIG_W, 2.9))
for j, (col, yl) in enumerate([("adv_vs_F1_pct", "Cost reduction of P7\nvs binary F1 (%)"), ("adv_vs_best_pct", "Cost reduction of P7\nvs best competitor (%)")]):
    ax = axes[j]
    for t in TAGS:
        for sc in SCORERS:
            s = SW[(SW.dataset == t) & (SW.scorer == sc)].sort_values("ratio_median_amount_to_C_rev"); main = sc == "LightGBM"
            ax.plot(s.ratio_median_amount_to_C_rev, s[col], color=CL[t], marker="o" if main else None, lw=1.6 if main else .6, alpha=1 if main else .45, label=DS_NAME[t] if main else None)
            if main:
                m1 = s[s.lam == 1]; ax.scatter(m1.ratio_median_amount_to_C_rev, m1[col], s=40, facecolors="none", edgecolors="k", zorder=5, lw=.8)
    ax.axhline(0, color="k", lw=.8); ax.set_xscale("log"); ax.set_xlabel("Median test fraud amount / $C_{\\mathrm{rev}}$"); ax.set_ylabel(yl)
    panel(ax, f"({'ab'[j]})", -0.24)
h_, l_ = axes[0].get_legend_handles_labels(); h_.append(plt.Line2D([], [], marker="o", ls="", mfc="none", mec="k")); l_.append("main setting ($\\lambda$ = 1)")
fig.legend(h_, l_, loc="upper center", ncol=4, bbox_to_anchor=(0.5, 1.06), frameon=False); fig.tight_layout(); save(fig, "Figure_8", SW)

# ---------- Figure 9: cost-ratio thresholding with raw and isotonic probabilities
fig, axes = plt.subplots(1, 3, figsize=(FIG_W, 2.6)); rows = []
for k, t in enumerate(TAGS):
    ax = axes[k]; c_ = pd.read_csv(f"{DIRS['output']}/{t}_costratio.csv"); rows.append(c_.assign(dataset=t))
    for sc, col in [("FTT_ens", "#d62728"), ("XGBoost", "#1f77b4")]:
        for pn, ls, mk in [("raw", "--", "s"), ("isotonic", "-", "o")]:
            s = c_[(c_.scorer == sc) & (c_.probs == pn)]; ax.plot(s.r, s.FP, ls=ls, marker=mk, color=col, label=f"{SC_NAME[sc]}, {pn}")
    ax.axhline(int((DEC[t]["yte"] == 0).sum()), color="k", ls=":", lw=.8); ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("Cost ratio $r$"); ax.set_ylabel("False positives" if k == 0 else ""); panel(ax, f"({'abc'[k]}) {DS_NAME[t]}", -0.24)
h_, l_ = axes[0].get_legend_handles_labels(); fig.legend(h_, l_, loc="upper center", ncol=4, bbox_to_anchor=(0.5, 1.08), frameon=False)
fig.tight_layout(); save(fig, "Figure_9", pd.concat(rows))

## Step 20: Additional figures for the code and data repository
Figures that do not fit the 34-page limit of the journal: exploratory ULB
figures of the earlier version (scatter matrix, temporal view, split violins,
t-SNE, radar, V14-V17 interaction), fraud rate over time, amount
distributions, IEEE-CIS feature gain, PR and ROC curves, FT-Transformer seeds,
threshold curves, confusion matrices, reliability diagrams, score densities,
amount recall, cost-sensitivity heat maps and the calibration-sample-size
curves. Saved in `fig/extra/` with the same PDF/PNG/CSV convention.

In [ ]:
# ---- compatibility layer: the extra-figure code below uses these names
from sklearn.metrics import roc_curve
DS = {t: (f"{DIRS['dataset']}/{t}_decision.npz", SEEDS_OF[t], K_GRID[t]) for t in ["ieee", "ulb", "bs"]}
load = lambda t, name, seeds: load_pred(t, name)
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
FIG = f"{DIRS['fig']}/extra"
plt.rcParams.update({"font.size": 9, "axes.grid": True, "grid.alpha": 0.3, "savefig.dpi": 300,
                     "figure.dpi": 100, "legend.fontsize": 7.5, "axes.axisbelow": True})
BLUE, RED, GREY = "#6baed6", "#d62728", "#7f7f7f"
POL_STYLE = {"binary_F1": ("#7f7f7f", "s", "--"), "binary_bayes_amount": ("#8c564b", "D", "--"),
             "topK_review": ("#bcbd22", "v", ":"), "reject_option": ("#9467bd", "^", ":"),
             "threeway_fixed": ("#1f77b4", "o", "-."), "threeway_expected_loss": ("#2ca02c", "P", "-."),
             "threeway_bayes_budget": ("#d62728", "o", "-")}
MODEL_ORDER = ["LogReg", "MLP", "RandomForest", "XGBoost", "LightGBM", "CatBoost", "FTT_ens"]
MODEL_NAME = {"FTT_ens": "FT-Transformer (ens.)", "RandomForest": "Random forest"}

def panel(ax, s, dy=-0.18):
    ax.text(0.5, dy, s, transform=ax.transAxes, ha="center", va="top", fontsize=10, fontweight="bold")

def save(fig, name, data):
    fig.savefig(f"{FIG}/{name}.png", dpi=300, bbox_inches="tight")
    fig.savefig(f"{FIG}/{name}.pdf", bbox_inches="tight")
    (data if isinstance(data, pd.DataFrame) else pd.DataFrame(data)).to_csv(f"{FIG}/{name}.csv", index=False)
    plt.close(fig); print("saved", name)

LETTERS = "abcdefghijkl"
u = pd.read_csv(ULB_CSV).sort_values("Time", kind="mergesort").reset_index(drop=True)
UL, UF = u[u.Class == 0], u[u.Class == 1]

# fig01: scatter matrix of four discriminative components
feats = ["V17", "V14", "V12", "V10"]; Ls = UL.sample(4000, random_state=42)
fig, axes = plt.subplots(4, 4, figsize=(8.5, 8.5))
for i, fi in enumerate(feats):
    for j, fj in enumerate(feats):
        ax = axes[i, j]
        if i == j:
            bins = np.linspace(min(Ls[fi].min(), UF[fi].min()), max(Ls[fi].max(), UF[fi].max()), 50)
            ax.hist(Ls[fi], bins=bins, color=BLUE, alpha=.7, density=True)
            ax.hist(UF[fi], bins=bins, color=RED, alpha=.5, density=True)
        else:
            ax.scatter(Ls[fj], Ls[fi], s=2, c=BLUE, alpha=.4, rasterized=True)
            ax.scatter(UF[fj], UF[fi], s=4, c=RED, alpha=.6, rasterized=True)
        if j == 0: ax.set_ylabel(fi)
        if i == 3: ax.set_xlabel(fj)
fig.legend(handles=[mpatches.Patch(color=BLUE, label="Legitimate (4,000 sampled)"),
                    mpatches.Patch(color=RED, label=f"Fraudulent (all {len(UF)})")],
           loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.01))
fig.tight_layout(rect=(0, 0, 1, 0.98))
save(fig, "fig01_ulb_scatter_matrix", pd.concat([Ls[feats + ["Class"]], UF[feats + ["Class"]]]))

# fig02: temporal view (amount and V17)
th = u.Time / 3600
fig, (a1, a2) = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
for ax, col, yl in [(a1, "Amount", "Transaction amount (EUR)"), (a2, "V17", "V17")]:
    ax.scatter(th[u.Class == 0][::20], UL[col][::20], s=2, c=BLUE, alpha=.3, label="Legitimate (every 20th)",
               rasterized=True)
    ax.scatter(th[u.Class == 1], UF[col], s=8, c=RED, alpha=.8, label="Fraudulent", rasterized=True)
    ax.set_ylabel(yl)
a1.set_yscale("symlog", linthresh=1); a1.set_ylim(0, u.Amount.max() * 1.5)
a1.legend(markerscale=3, loc="upper left", framealpha=.9)
for b_ in [int(len(u) * .70), int(len(u) * .85)]:
    for ax in (a1, a2): ax.axvline(th[b_], color="k", ls=":", lw=1)
a2.set_xlabel("Time (hours since first transaction)")
panel(a1, "(a)", dy=-0.08); panel(a2, "(b)", dy=-0.22)
fig.tight_layout()
save(fig, "fig02_ulb_temporal", pd.DataFrame({"hours": th, "Amount": u.Amount, "V17": u.V17, "Class": u.Class})
     .iloc[np.r_[np.where(u.Class == 0)[0][::20], np.where(u.Class == 1)[0]]])

# fig03: split violins
vf = ["V17", "V14", "V12", "V10"]
fig, ax = plt.subplots(figsize=(8, 4.2))
for k, f in enumerate(vf):
    for cls, c, side in [(0, BLUE, -1), (1, RED, 1)]:
        v = ax.violinplot([u[u.Class == cls][f].values], positions=[k + 1], widths=.85, showmedians=True)
        for b in v["bodies"]:
            b.set_facecolor(c); b.set_edgecolor(c); b.set_alpha(.6)
            path = b.get_paths()[0].vertices; m = np.mean(path[:, 0])
            path[:, 0] = np.clip(path[:, 0], m, np.inf) if side == 1 else np.clip(path[:, 0], -np.inf, m)
        for part in ["cmedians", "cmins", "cmaxes", "cbars"]:
            v[part].set_color(c)
ax.set_xticks(range(1, len(vf) + 1)); ax.set_xticklabels(vf); ax.set_ylabel("Standardised component value")
ax.legend(handles=[mpatches.Patch(color=BLUE, label="Legitimate"), mpatches.Patch(color=RED, label="Fraudulent")])
fig.tight_layout()
save(fig, "fig03_ulb_violin", u.groupby("Class")[vf].describe().T.reset_index())

# fig04: t-SNE of a class-balanced subsample
from sklearn.manifold import TSNE
sub = pd.concat([UL.sample(3000, random_state=42), UF])
Xs = StandardScaler().fit_transform(sub.drop(columns=["Class", "Time"]))
try:
    emb = TSNE(n_components=2, perplexity=35, random_state=42, init="pca", max_iter=1000).fit_transform(Xs)
except TypeError:
    emb = TSNE(n_components=2, perplexity=35, random_state=42, init="pca", n_iter=1000).fit_transform(Xs)
fig, ax = plt.subplots(figsize=(6, 5.2)); m_ = sub.Class.values == 0
ax.scatter(emb[m_, 0], emb[m_, 1], s=4, c=BLUE, alpha=.5, label="Legitimate (3,000 sampled)", rasterized=True)
ax.scatter(emb[~m_, 0], emb[~m_, 1], s=8, c=RED, alpha=.8, label=f"Fraudulent ({len(UF)})", rasterized=True)
ax.set_xlabel("t-SNE dimension 1"); ax.set_ylabel("t-SNE dimension 2"); ax.legend()
fig.tight_layout()
save(fig, "fig04_ulb_tsne", pd.DataFrame({"tsne1": emb[:, 0], "tsne2": emb[:, 1], "Class": sub.Class.values}))

# fig05: radar profile of class means
rf = ["V12", "V14", "V17", "V4", "V11", "V10"]
ang = np.linspace(0, 2 * np.pi, len(rf), endpoint=False).tolist(); ang += ang[:1]
fig, ax = plt.subplots(figsize=(5.5, 5), subplot_kw=dict(polar=True))
rad = {}
for s_, c, nm in [(UL, BLUE, "Legitimate"), (UF, RED, "Fraudulent")]:
    v = [s_[f].mean() for f in rf]; rad[nm] = v; v = v + v[:1]
    ax.plot(ang, v, color=c, lw=2, label=nm); ax.fill(ang, v, color=c, alpha=.25)
ax.set_xticks(ang[:-1]); ax.set_xticklabels(rf); ax.legend(loc="lower left", bbox_to_anchor=(-.15, -.12))
fig.tight_layout()
save(fig, "fig05_ulb_radar", pd.DataFrame({"feature": rf, **rad}))

# fig06: V14-V17 interaction
fig, ax = plt.subplots(figsize=(6, 5)); Ls = UL.sample(8000, random_state=42)
ax.scatter(Ls.V14, Ls.V17, s=3, c=BLUE, alpha=.35, label="Legitimate (8,000 sampled)", rasterized=True)
ax.scatter(UF.V14, UF.V17, s=10, c=RED, alpha=.7, label="Fraudulent", rasterized=True)
ax.set_xlabel("V14"); ax.set_ylabel("V17"); ax.legend(markerscale=2)
fig.tight_layout()
save(fig, "fig06_ulb_interaction", pd.concat([Ls[["V14", "V17", "Class"]], UF[["V14", "V17", "Class"]]]))
del u, UL, UF; gc.collect()

from sklearn.metrics import roc_curve
TAGS = list(DS)
DD = {t: np.load(DS[t][0]) for t in TAGS}
YT = {t: DD[t]["yte"] for t in TAGS}; YV = {t: DD[t]["yva"] for t in TAGS}
AT = {t: DD[t]["amt_te"].astype(float) for t in TAGS}

# fig07: fraud rate over time with split boundaries
mt = np.load(f"{DIRS['dataset']}/ieee_meta.npz")
u_ = pd.read_csv(ULB_CSV, usecols=["Time", "Class"]).sort_values("Time", kind="mergesort")
b_ = pd.read_csv(BANKSIM_CSV, usecols=["step", "fraud"]).sort_values("step", kind="mergesort")
series = {"ieee": (mt["t"] / 86400, mt["y"], "Days since first transaction", 60),
          "ulb":  (u_.Time.values / 3600, u_.Class.values, "Hours since first transaction", 48),
          "bs":   (b_.step.values.astype(float), b_.fraud.values, "Simulation step (day)", 60)}
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4)); rows = []
for k, (t, (x, y_, xl, nb)) in enumerate(series.items()):
    ax = axes[k]; ed = np.linspace(x.min(), x.max(), nb + 1); bi = np.clip(np.digitize(x, ed) - 1, 0, nb - 1)
    cnt = np.bincount(bi, minlength=nb); fr = np.bincount(bi, weights=y_, minlength=nb)
    mid = (ed[:-1] + ed[1:]) / 2; rate = 100 * fr / np.maximum(cnt, 1)
    ax.plot(mid, rate, color=RED, lw=1.5)
    n_ = len(x)
    for cut in [int(n_ * SPLIT[0]), int(n_ * (SPLIT[0] + SPLIT[1]))]:
        ax.axvline(np.sort(x)[cut], color="k", ls=":", lw=1)
    ax.set_xlabel(xl); ax.set_ylabel("Fraud rate (%)" if k == 0 else ""); panel(ax, f"({LETTERS[k]}) {DS_NAME[t]}", -0.2)
    rows += [dict(dataset=t, bin_mid=m, n=c, fraud=f, rate_pct=r) for m, c, f, r in zip(mid, cnt, fr, rate)]
fig.tight_layout(); save(fig, "fig07_temporal_fraud_rate", rows)

# fig08: amount distributions by class (ECDF, log scale)
amt_full = {"ieee": (mt["amt"], mt["y"]),
            "ulb": tuple(pd.read_csv(ULB_CSV, usecols=["Amount", "Class"]).values.T),
            "bs": tuple(pd.read_csv(BANKSIM_CSV, usecols=["amount", "fraud"]).values.T)}
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4)); rows = []
for k, (t, (am, y_)) in enumerate(amt_full.items()):
    ax = axes[k]; am = np.maximum(am.astype(float), 0.01); y_ = y_.astype(int)
    for cls, c, nm in [(0, BLUE, "Legitimate"), (1, RED, "Fraudulent")]:
        v = np.sort(am[y_ == cls]); e = np.arange(1, len(v) + 1) / len(v)
        step_ = max(1, len(v) // 2000); ax.plot(v[::step_], e[::step_], color=c, lw=1.8, label=nm)
        rows.append(dict(dataset=t, cls=nm, n=len(v), median=np.median(v), mean=v.mean(), p90=np.quantile(v, .9),
                         top10pct_share_of_total=v[int(.9 * len(v)):].sum() / v.sum()))
    ax.set_xscale("log"); ax.set_xlabel("Transaction amount (log scale)")
    ax.set_ylabel("Cumulative share" if k == 0 else ""); ax.legend(loc="lower right")
    panel(ax, f"({LETTERS[k]}) {DS_NAME[t]}", -0.2)
fig.tight_layout(); save(fig, "fig08_amount_distribution", rows)
del u_, b_; gc.collect()

# fig09: IEEE-CIS feature gain
cg = pd.read_csv(f"{DIRS['output']}/ieee_cumulative_gain.csv")
fi = pd.read_csv(f"{DIRS['output']}/ieee_feature_importance.csv").head(20)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.plot(cg.k, cg.cum_gain_share, color=RED, lw=2)
a1.axvline(TOPK_FTT, color="k", ls=":", lw=1)
a1.annotate(f"top-{TOPK_FTT}: {cg.cum_gain_share[TOPK_FTT-1]:.3f}", (TOPK_FTT, cg.cum_gain_share[TOPK_FTT-1]),
            xytext=(15, -25), textcoords="offset points", arrowprops=dict(arrowstyle="->"))
a1.set_xlabel("Number of features (ranked by gain)"); a1.set_ylabel("Cumulative gain share")
a2.barh(fi.feature[::-1], fi.gain[::-1], color=BLUE, edgecolor="k", lw=.4)
a2.set_xlabel("LightGBM gain"); a2.tick_params(axis="y", labelsize=7)
panel(a1, "(a)"); panel(a2, "(b)")
fig.tight_layout(); save(fig, "fig09_ieee_feature_gain", fi.assign(cum_share_at_150=cg.cum_gain_share[TOPK_FTT-1]))

# fig10: PR (top row) and ROC (bottom row) curves
fig, axes = plt.subplots(2, 3, figsize=(13, 8.4)); rows = []
for k, t in enumerate(TAGS):
    a1, a2 = axes[0, k], axes[1, k]; y_ = YT[t]
    for nm in MODEL_ORDER:
        pt = load(t, nm, DS[t][1])[1]; lw, z = (2.2, 5) if nm == "FTT_ens" else (1.2, 2)
        pr, rc, _ = precision_recall_curve(y_, pt); fpr, tpr, _ = roc_curve(y_, pt)
        ap, au = average_precision_score(y_, pt), roc_auc_score(y_, pt)
        a1.plot(rc, pr, lw=lw, zorder=z, label=f"{MODEL_NAME.get(nm, nm)} ({ap:.3f})")
        a2.plot(fpr, tpr, lw=lw, zorder=z, label=f"{MODEL_NAME.get(nm, nm)} ({au:.3f})")
        rows.append(dict(dataset=t, model=nm, AUPRC=ap, AUROC=au))
    a1.axhline(y_.mean(), color="gray", ls="--", lw=.8)
    a2.plot([0, 1], [0, 1], color="gray", ls="--", lw=.8)
    a1.set_xlabel("Recall"); a1.set_ylabel("Precision"); a1.legend(fontsize=6.5, loc="lower left", framealpha=.9)
    a2.set_xlabel("False positive rate"); a2.set_ylabel("True positive rate"); a2.legend(fontsize=6.5, loc="lower right")
    panel(a1, f"({LETTERS[k]}) {DS_NAME[t]}, PR"); panel(a2, f"({LETTERS[k+3]}) {DS_NAME[t]}, ROC")
fig.tight_layout(h_pad=3); save(fig, "fig10_pr_roc", rows)

# fig11: FT-Transformer seed-level test AUPRC versus tree ensembles
L1 = pd.read_csv(f"{DIRS['output']}/ieee_ftt_training_log.csv").assign(tag="ieee", name="FTT")
L2 = pd.read_csv(f"{DIRS['output']}/ulb_bs_ftt_training_log.csv")
LL = pd.concat([L1[["tag", "name", "seed", "test_auprc"]], L2[["tag", "name", "seed", "test_auprc"]]])
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6)); rows = []
for k, t in enumerate(TAGS):
    ax = axes[k]; ph = pd.read_csv(f"{DIRS['output']}/{t}_phase1_results.csv").set_index("model")
    groups = [("FTT", "FT-T seeds")] + ([("ABL-focalL1", "1 block"), ("ABL-bceL2", "BCE loss")] if t == "ulb" else [])
    for gi, (g, lab) in enumerate(groups):
        v = LL[(LL.tag == t) & (LL.name == g)].test_auprc.values
        ax.scatter(np.full(len(v), gi) + np.linspace(-.08, .08, len(v)), v, color=RED if g == "FTT" else GREY, zorder=3)
        ax.errorbar(gi + .2, v.mean(), yerr=v.std(ddof=1), fmt="_", color="k", capsize=4)
        rows += [dict(dataset=t, group=g, seed_auprc=x) for x in v]
    for nm, c in [("XGBoost", "#1f77b4"), ("LightGBM", "#2ca02c"), ("CatBoost", "#ff7f0e"), ("FTT_ens", RED)]:
        ax.axhline(ph.loc[nm, "AUPRC"], color=c, ls="--" if nm != "FTT_ens" else "-", lw=1,
                   label=f"{MODEL_NAME.get(nm, nm)} {ph.loc[nm, 'AUPRC']:.3f}")
        rows.append(dict(dataset=t, group=nm, seed_auprc=ph.loc[nm, "AUPRC"]))
    ax.set_xticks(range(len(groups))); ax.set_xticklabels([g[1] for g in groups]); ax.set_xlim(-.5, len(groups) - .3)
    ax.set_ylabel("Test AUPRC" if k == 0 else ""); ax.legend(fontsize=6.5, loc="lower right")
    panel(ax, f"({LETTERS[k]}) {DS_NAME[t]}", -0.14)
fig.tight_layout(); save(fig, "fig11_ftt_seed_auprc", rows)

# fig12: precision, recall and F1 versus threshold (FT-Transformer ensemble, isotonic)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6)); rows = []
for k, t in enumerate(TAGS):
    ax = axes[k]; pv, pt = load(t, "FTT_ens", DS[t][1]); pc = iso(pv, YV[t], pt)
    th_ = f1_threshold(YV[t], iso(pv, YV[t], pv)); y_ = YT[t]
    for th in np.linspace(0.01, 0.99, 99):
        f = pc >= th; tp = (f & (y_ == 1)).sum(); P_ = tp / max(f.sum(), 1); R_ = tp / y_.sum()
        rows.append(dict(dataset=t, threshold=th, precision=P_, recall=R_, f1=2 * P_ * R_ / max(P_ + R_, 1e-12)))
    r_ = pd.DataFrame([r for r in rows if r["dataset"] == t])
    ax.plot(r_.threshold, r_.precision, "b--", label="Precision"); ax.plot(r_.threshold, r_.recall, "g-", label="Recall")
    ax.plot(r_.threshold, r_.f1, "r-", lw=2, label="F1")
    ax.axvline(th_, color="k", ls=":", label=f"validation threshold {th_:.3f}")
    ax.set_xlabel("Threshold on calibrated probability"); ax.set_ylabel("Score" if k == 0 else "")
    ax.legend(loc="lower center"); panel(ax, f"({LETTERS[k]}) {DS_NAME[t]}", -0.2)
fig.tight_layout(); save(fig, "fig12_threshold_curves", rows)

# fig13: confusion matrices at the validation F1 threshold (FT-Transformer ensemble, isotonic)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6)); rows = []
for k, t in enumerate(TAGS):
    ax = axes[k]; pv, pt = load(t, "FTT_ens", DS[t][1]); pc = iso(pv, YV[t], pt)
    th_ = f1_threshold(YV[t], iso(pv, YV[t], pv))
    tn, fp, fn, tp = confusion_matrix(YT[t], (pc >= th_).astype(int)).ravel(); M = np.array([[tn, fp], [fn, tp]])
    im = ax.imshow(M, cmap="Blues", norm=matplotlib.colors.LogNorm(vmin=max(M.min(), 1), vmax=M.max()))
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{M[i, j]:,}", ha="center", va="center", fontsize=11,
                    color="white" if M[i, j] > np.sqrt(M.max()) * 3 else "black")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Legitimate", "Fraud"]); ax.set_yticks([0, 1])
    ax.set_yticklabels(["Legitimate", "Fraud"]); ax.set_xlabel("Predicted"); ax.set_ylabel("Actual" if k == 0 else "")
    ax.grid(False); panel(ax, f"({LETTERS[k]}) {DS_NAME[t]}", -0.2)
    rows.append(dict(dataset=t, threshold=th_, TN=tn, FP=fp, FN=fn, TP=tp))
fig.tight_layout(); save(fig, "fig13_confusion", rows)

from sklearn.calibration import calibration_curve
# fig14: reliability diagrams of the FT-Transformer ensemble
fig, axes = plt.subplots(1, 3, figsize=(12, 3.9)); rows = []
for k, t in enumerate(TAGS):
    ax = axes[k]; pv, pt = load(t, "FTT_ens", DS[t][1]); yv_, y_ = YV[t], YT[t]
    cands = {"Raw": (pt, GREY), "Temperature": (temp(pv, yv_, pt), "#2ca02c"),
             "Platt": (platt(pv, yv_, pt), "#1f77b4"), "Isotonic": (iso(pv, yv_, pt), RED)}
    ax.plot([1e-5, 1], [1e-5, 1], "k--", lw=1, label="Perfect calibration")
    for nm, (q, c) in cands.items():
        fr, mp_ = calibration_curve(y_, q, n_bins=12, strategy="quantile")
        ok = (fr > 0) & (mp_ > 0)
        ax.plot(mp_[ok], fr[ok], "o-", color=c, ms=3.5, label=f"{nm} (ECE {ece(y_, q):.4f})")
        rows += [dict(dataset=t, calibrator=nm, mean_pred=a_, frac_pos=b_) for a_, b_ in zip(mp_, fr)]
    ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Observed fraud fraction" if k == 0 else ""); ax.legend(fontsize=6.5, loc="upper left")
    panel(ax, f"({LETTERS[k]}) {DS_NAME[t]}", -0.2)
fig.tight_layout(); save(fig, "fig14_reliability", rows)

# fig15: class-conditional distribution of calibrated probabilities
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6)); rows = []
bins = np.linspace(0, 1, 41)
for k, t in enumerate(TAGS):
    ax = axes[k]; pv, pt = load(t, "FTT_ens", DS[t][1]); pc = iso(pv, YV[t], pt); y_ = YT[t]
    h0, _ = np.histogram(pc[y_ == 0], bins); h1, _ = np.histogram(pc[y_ == 1], bins)
    ax.hist(pc[y_ == 0], bins=bins, color=BLUE, alpha=.7, label="Legitimate", log=True)
    ax.hist(pc[y_ == 1], bins=bins, color=RED, alpha=.7, label="Fraudulent", log=True)
    ax.set_xlabel("Calibrated fraud probability"); ax.set_ylabel("Count (log scale)" if k == 0 else "")
    ax.legend(); panel(ax, f"({LETTERS[k]}) {DS_NAME[t]}", -0.2)
    rows += [dict(dataset=t, bin_lo=lo, legit=a_, fraud=b_) for lo, a_, b_ in zip(bins[:-1], h0, h1)]
fig.tight_layout(); save(fig, "fig15_score_density", rows)

# fig16: cost-ratio thresholding, false positives with raw versus isotonic probabilities
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6)); rows = []
for k, t in enumerate(TAGS):
    ax = axes[k]; c_ = pd.read_csv(f"{DIRS['output']}/{t}_costratio.csv"); rows.append(c_.assign(dataset=t))
    for sc_, col in [("FTT_ens", RED), ("XGBoost", "#1f77b4")]:
        for pn, ls, mk in [("raw", "--", "s"), ("isotonic", "-", "o")]:
            s_ = c_[(c_.scorer == sc_) & (c_.probs == pn)]
            ax.plot(s_.r, s_.FP, ls=ls, marker=mk, color=col, ms=4,
                    label=f"{MODEL_NAME.get(sc_, sc_)}, {pn}")
    n_leg = int((YT[t] == 0).sum()); ax.axhline(n_leg, color="k", ls=":", lw=1)
    ax.text(1, n_leg * 0.6, "all legitimate test transactions", fontsize=7)
    ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("Cost ratio r = C_FN / C_FP")
    ax.set_ylabel("False positives" if k == 0 else "")
    panel(ax, f"({LETTERS[k]}) {DS_NAME[t]}", -0.2)
h_, l_ = axes[0].get_legend_handles_labels()
fig.legend(h_, l_, loc="upper center", ncol=4, bbox_to_anchor=(0.5, 1.06))
fig.tight_layout(); save(fig, "fig16_costratio", pd.concat(rows))

MIDK = {t: DS[t][2][len(DS[t][2]) // 2] for t in TAGS}
POLD = {t: pd.read_csv(f"{DIRS['output']}/{t}_policies.csv") for t in TAGS}

# fig18 / fig19: total cost and amount recall versus review budget
for fname, col, ylab in [("fig19_amount_recall", "amount_recall", "Fraud amount recall")]:
    fig, axes = plt.subplots(2, 3, figsize=(13, 8)); rows = []
    for i, sc_ in enumerate(["LightGBM", "FTT_ens"]):
        for k, t in enumerate(TAGS):
            ax = axes[i, k]; P_ = POLD[t][(POLD[t].calib == "isotonic") & (POLD[t].scorer == sc_)]
            for p_ in POLICIES:
                s_ = P_[P_.policy == p_].sort_values("K"); c, mk, ls = POL_STYLE[p_]
                ax.plot(s_.K, s_[col], ls=ls, marker=mk, color=c, ms=4, lw=2.2 if p_ == PROPOSED else 1.2,
                        zorder=5 if p_ == PROPOSED else 2, label=POL_NAME[p_])
                rows += [dict(scorer=sc_, dataset=t, policy=p_, K=kk, value=v) for kk, v in zip(s_.K, s_[col])]
            ax.set_xscale("log"); ax.set_xlabel("Review budget K")
            ax.set_xticks(DS[t][2]); ax.set_xticklabels([f"{v:,}" for v in DS[t][2]]); ax.minorticks_off()
            if col == "cost":
                ax.set_yscale("log")
            ax.set_ylabel(f"{ylab}\n({MODEL_NAME.get(sc_, sc_)})" if k == 0 else "")
            panel(ax, f"({LETTERS[3*i+k]}) {DS_NAME[t]}, {MODEL_NAME.get(sc_, sc_)}", -0.2)
    h_, l_ = axes[0, 0].get_legend_handles_labels()
    fig.legend(h_, l_, loc="upper center", ncol=4, bbox_to_anchor=(0.5, 1.03))
    fig.tight_layout(rect=(0, 0, 1, 0.97), h_pad=3); save(fig, fname, rows)

# fig22: cost sensitivity, relative gain of the proposed policy over the cheapest competitor (LightGBM)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.7)); rows = []
for k, t in enumerate(TAGS):
    ax = axes[k]; S_ = pd.read_csv(f"{DIRS['output']}/{t}_cost_sensitivity.csv")
    S_ = S_[(S_.calib == "isotonic") & (S_.scorer == "LightGBM")]
    M = S_.pivot(index="C_rev", columns="C_blk", values="rel_gain_pct")
    lim = max(abs(M.values).max(), 1)
    im = ax.imshow(M.values, cmap="RdBu", vmin=-lim, vmax=lim, origin="lower")
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            ax.text(j, i, f"{M.values[i, j]:+.1f}%", ha="center", va="center", fontsize=8)
    ax.set_xticks(range(M.shape[1])); ax.set_xticklabels(M.columns); ax.set_yticks(range(M.shape[0]))
    ax.set_yticklabels(M.index); ax.set_xlabel("False-block cost C_blk"); ax.set_ylabel("Review cost C_rev" if k == 0 else "")
    ax.grid(False); fig.colorbar(im, ax=ax, fraction=.046, label="Gain over best competitor (%)" if k == 2 else "")
    panel(ax, f"({LETTERS[k]}) {DS_NAME[t]}, K = {MIDK[t]:,}", -0.22)
    rows.append(S_.assign(dataset=t))
fig.tight_layout(); save(fig, "fig22_cost_sensitivity", pd.concat(rows))

# fig23: advantage over binary_F1 versus number of validation fraud cases used for calibration
S = pd.read_csv(f"{DIRS['output']}/calibration_size_summary.csv")
fig, axes = plt.subplots(1, 3, figsize=(13, 3.9))
for k, t in enumerate(["ieee", "bs", "ulb"]):
    ax = axes[k]
    for sc_, ls in [("LightGBM", "-"), ("FTT_ens", "--")]:
        for cb, c in [("isotonic", RED), ("platt", "#1f77b4")]:
            s_ = S[(S.dataset == t) & (S.scorer == sc_) & (S.calib == cb)].sort_values("n_pos")
            ax.plot(s_.n_pos, s_.adv_vs_F1_mean, ls=ls, marker="o", ms=3.5, color=c,
                    label=f"{MODEL_NAME.get(sc_, sc_)}, {cb}")
            ax.fill_between(s_.n_pos, s_.adv_vs_F1_q25, s_.adv_vs_F1_q75, color=c, alpha=.08)
    ax.axhline(0, color="k", lw=1)
    if t != "ulb": ax.axvline(56, color=GREY, ls=":", lw=1); ax.text(58, ax.get_ylim()[0], " ULB size", fontsize=7, va="bottom")
    ax.set_xscale("log"); ax.set_xlabel("Validation fraud cases used for calibration")
    ax.set_xticks(CAL_SIZE_GRID[t]); ax.set_xticklabels([f"{v:,}" for v in CAL_SIZE_GRID[t]], fontsize=7); ax.minorticks_off()
    ax.set_ylabel("Cost reduction vs binary F1 (%)" if k == 0 else ""); ax.legend(fontsize=6.5)
    panel(ax, f"({LETTERS[k]}) {DS_NAME[t]}, K = {MIDK[t]:,}", -0.2)
fig.tight_layout(); save(fig, "fig23_calibration_size", S)
print("extra figures:", len(os.listdir(FIG)))

## Step 21: Temporal block bootstrap, Holm correction and effect sizes
Revision analysis. The transaction-level bootstrap of Step 11 treats test
transactions as independent. Here the per-transaction costs of each policy are
resampled in **blocks of consecutive transactions** (moving-block bootstrap,
`N_BLOCKS` blocks per test window, `N_BOOT_BLOCK` resamples), which keeps
short-range dependence in time. Two-sided bootstrap p-values are corrected with
the **Holm** procedure within each family (dataset x budget mode: all
competitor x scorer x budget cells). Effect sizes are relative cost reductions
of P7 with 95% block-bootstrap intervals; P7 vs P2 and P7 vs P6 are the
pre-specified primary comparisons.

`decide_arr` re-implements `decide` on arrays so that later steps can use other
amounts, costs, analyst accuracies, subsets of the test window and other time
windows; it is checked against `decide` first.

In [ ]:
t0 = time.time()

def decide_arr(name, pv, pt, av, at, th, K=None, mode="offline", per_va=None, per_te=None,
               c_rev=C_REV, c_blk=C_BLK, q=1.0, cblk_va=None, comp=None):
    """Array version of decide(). c_blk may be a per-transaction array (test); cblk_va is its validation counterpart.
    comp: optional function(name, p, a, th, c_rev, c_blk, q) replacing components()."""
    comp = comp or components
    blk0, cand, prio = comp(name, pt, at, th, c_rev, c_blk, q)
    if not cand.any() or K is None: return blk0, np.zeros_like(blk0)
    kw = {}
    if mode != "offline":
        _, cand_v, prio_v = comp(name, pv, av, th, c_rev, c_blk if cblk_va is None else cblk_va, q)
        n_va, n_te = len(pv), len(pt)
        if mode.startswith("online"):
            kw["tau"] = kth_value(prio_v, cand_v, round(K * n_va / n_te))
        else:
            P_te, P_va = len(np.unique(per_te)), len(np.unique(per_va))
            k_per = max(int(round(K / P_te)), 1)
            kw.update(per=per_te, k_per=k_per, tau0=kth_value(prio_v, cand_v, k_per * P_va))
    rev = select(mode, prio, cand, K, **kw)
    return blk0 & ~rev, rev

def arrays(t, sc, calib="isotonic"):
    C, D = CAL[t][sc][calib], DEC[t]
    return dict(pv=C["pv"], pt=C["pt"], av=D["amt_va"].astype(float), at=D["amt_te"].astype(float), th=C["th"],
                per_va=D["per_va"], per_te=D["per_te"])

# consistency check against decide()
for t in DEC:
    A = arrays(t, "LightGBM")
    for m in ["offline", "online_capped", "period_adaptive"]:
        for p in POLICIES:
            b1, r1 = decide(p, t, "LightGBM", "isotonic", MID_K[t], m)
            b2, r2 = decide_arr(p, A["pv"], A["pt"], A["av"], A["at"], A["th"], MID_K[t], m, A["per_va"], A["per_te"])
            assert (b1 == b2).all() and (r1 == r2).all(), (t, m, p)
print("decide_arr matches decide")

def block_sums(costs, n_blocks=N_BLOCKS):
    """Circular moving-block sums: row s = total cost of the L consecutive transactions starting at s."""
    M = np.stack(costs, 1); n = len(M); L = max(n // n_blocks, 1)
    cs = np.vstack([np.zeros((1, M.shape[1])), np.cumsum(np.vstack([M, M[:L]]), 0)])
    return cs[L:n + L] - cs[:n], L

def block_boot(costs, ref_idx, n_boot=N_BOOT_BLOCK, seed=7, n_blocks=N_BLOCKS):
    """Moving-block bootstrap of total-cost differences (competitor - ref) for every column of `costs`."""
    S, L = block_sums(costs, n_blocks); n = len(S); nb = int(np.ceil(n / L))
    rng = np.random.default_rng(seed); tot = np.zeros((n_boot, S.shape[1]))
    for i in range(0, n_boot, 500):
        st = rng.integers(0, n, (min(500, n_boot - i), nb)); tot[i:i + len(st)] = S[st].sum(1) * (n / (nb * L))
    obs = np.stack(costs, 1).sum(0); out = []
    for k in range(S.shape[1]):
        if k == ref_idx: continue
        d = tot[:, k] - tot[:, ref_idx]; rel = 100 * d / tot[:, k]
        p = min(1.0, 2 * min(((d <= 0).sum() + 1) / (n_boot + 1), ((d >= 0).sum() + 1) / (n_boot + 1)))
        out.append(dict(col=k, diff=obs[k] - obs[ref_idx], rel_pct=100 * (obs[k] - obs[ref_idx]) / obs[k],
                        ci_lo=np.percentile(d, 2.5), ci_hi=np.percentile(d, 97.5),
                        rel_lo=np.percentile(rel, 2.5), rel_hi=np.percentile(rel, 97.5), p=p))
    return out

def holm(p, alpha=ALPHA):
    p = np.asarray(p, float); o = np.argsort(p); m = len(p); rej = np.zeros(m, bool)
    for r, i in enumerate(o):
        if p[i] <= alpha / (m - r): rej[i] = True
        else: break
    return rej

MODES_B = ["offline", "online_capped", "period_adaptive"]; rows = []
for t in DEC:
    y1 = DEC[t]["yte"] == 1
    for sc in SCORERS:
        A = arrays(t, sc)
        for K in K_GRID[t]:
            for m in MODES_B:
                pols = POLICIES if m == "offline" else BUDGETED
                costs = [tx_cost(*decide_arr(p, A["pv"], A["pt"], A["av"], A["at"], A["th"], K, m, A["per_va"], A["per_te"]), y1, A["at"])
                         for p in pols]
                for r in block_boot(costs, pols.index(PROPOSED)):
                    rows.append(dict(dataset=t, scorer=sc, K=K, mode=m, competitor=pols[r.pop("col")], **r))
BB = pd.DataFrame(rows)
BB["holm_reject"] = False
for key, g in BB.groupby(["dataset", "mode"]):
    BB.loc[g.index, "holm_reject"] = holm(g.p.values)
BB["verdict_holm"] = np.where(BB.holm_reject, np.where(BB["diff"] > 0, "better", "worse"), "ns")
BB["verdict_raw"] = np.where(BB.ci_lo > 0, "better", np.where(BB.ci_hi < 0, "worse", "ns"))
BB.to_csv(f"{DIRS['output']}/block_bootstrap_holm.csv", index=False)

for (t, m), g in BB.groupby(["dataset", "mode"]):
    v = g.pivot_table(index="competitor", columns="verdict_holm", values="diff", aggfunc="count", fill_value=0)
    for comp in v.index:
        for vd in v.columns: note(f"{t}_blockholm_{m}_{POL_SHORT[comp]}_{vd}", int(v.loc[comp, vd]), "block_bootstrap_holm.csv")
    print(f"\n== {DS_NAME[t]} | {m} | Holm-corrected block bootstrap (better/ns/worse)"); print(v)

# effect sizes at the middle budget (primary comparisons first)
ES = BB[BB.K == BB.dataset.map(MID_K)].copy()
ES.to_csv(f"{DIRS['output']}/effect_sizes_mid_budget.csv", index=False)
for _, r in ES[ES.competitor.isin(PRIMARY) & (ES["mode"] == "offline")].iterrows():
    k = f"{r.dataset}_es_{r.scorer}_P7_vs_{POL_SHORT[r.competitor]}"
    note(k + "_pct", r.rel_pct, "effect_sizes_mid_budget.csv"); note(k + "_lo", r.rel_lo, "effect_sizes_mid_budget.csv")
    note(k + "_hi", r.rel_hi, "effect_sizes_mid_budget.csv")
print(ES[ES.competitor.isin(PRIMARY) & (ES["mode"] == "offline")][["dataset", "scorer", "competitor", "rel_pct", "rel_lo", "rel_hi", "verdict_holm"]].round(2))
print(f"{time.time()-t0:.0f} s")

## Step 22: Tuned competitors and decomposition of the advantage of P7
Revision analysis for the fairness of the comparison.
- **Tuned thresholds.** P5 and P6 use the fixed blocking threshold
  `THETA_HIGH = 0.9`. Here `theta_H` is chosen from `THETA_GRID` to minimise
  the cost on the validation partition (with the budget scaled to its size),
  for every scorer and budget. P1 is also given a cost-tuned threshold (P1-c):
  the probability threshold that minimises validation cost.
- **Decomposition.** P6 and P7 differ in two components: the block rule
  (fixed `theta_H` vs expected cost) and the review selection (fill the budget
  by expected loss vs review only positive gains, ranked by gain). The 2 x 2
  design adds A1 (fixed `theta_H` block, gain-filtered reviews) and A2
  (expected-cost block, budget filled by expected loss), so that the cost
  difference P6 - P7 can be split into the two effects.

In [ ]:
t0 = time.time()

def make_comp(theta=THETA_HIGH):
    """components() with a configurable theta_H and the two decomposition policies A1, A2."""
    def comp(name, p, a, th, c_rev=C_REV, c_blk=C_BLK, q=1.0):
        none = np.zeros_like(p, bool)
        if name in ("threeway_fixed", "threeway_expected_loss"):
            blk = p >= theta; return blk, ~blk, (p if name == "threeway_fixed" else p * a)
        if name == "A1_fixedblock_gain":            # fixed theta_H block, review only if p*a - C_rev > 0
            blk = p >= theta; g = p * a - c_rev - (1 - q) * p * a; return blk, ~blk & (g > 0), g
        if name == "A2_bayesblock_fill":            # expected-cost block, fill budget by expected loss
            return (1 - p) * c_blk < p * a, np.ones_like(p, bool), p * a
        return components(name, p, a, th, c_rev, c_blk, q)
    return comp

def val_cost(name, A, yv, K, comp, th=None):
    Kv = int(round(K * len(A["pv"]) / len(A["pt"])))
    b, r = decide_arr(name, A["pv"], A["pv"], A["av"], A["av"], A["th"] if th is None else th, Kv, "offline", comp=comp)
    return tx_cost(b, r, yv == 1, A["av"]).sum()

rows = []
for t in DEC:
    yv, y1 = DEC[t]["yva"], DEC[t]["yte"] == 1
    for sc in SCORERS:
        A = arrays(t, sc)
        # P1-c: single probability threshold minimising validation cost
        grid = np.unique(np.quantile(A["pv"], np.linspace(0.5, 0.9999, 400)))
        vc = [tx_cost(A["pv"] >= g, np.zeros_like(yv, bool), yv == 1, A["av"]).sum() for g in grid]
        th_c = float(grid[int(np.argmin(vc))])
        c = tx_cost(A["pt"] >= th_c, np.zeros_like(y1), y1, A["at"]).sum()
        for K in K_GRID[t]:
            rows.append(dict(dataset=t, scorer=sc, K=K, policy="P1-c (cost-tuned threshold)", theta=th_c, cost=c))
            for name, lab in [("threeway_fixed", "P5"), ("threeway_expected_loss", "P6"), ("A1_fixedblock_gain", "A1")]:
                best = min(THETA_GRID, key=lambda th_: val_cost(name, A, yv, K, make_comp(th_)))
                for th_, tag in [(THETA_HIGH, ""), (best, "-tuned")]:
                    b, r = decide_arr(name, A["pv"], A["pt"], A["av"], A["at"], A["th"], K, "offline", comp=make_comp(th_))
                    rows.append(dict(dataset=t, scorer=sc, K=K, policy=lab + tag, theta=th_, cost=tx_cost(b, r, y1, A["at"]).sum()))
            for name, lab in [("A2_bayesblock_fill", "A2"), (PROPOSED, "P7"), ("binary_bayes_amount", "P2"), ("binary_F1", "P1"),
                              ("topK_review", "P3"), ("reject_option", "P4")]:
                b, r = decide_arr(name, A["pv"], A["pt"], A["av"], A["at"], A["th"], K, "offline", comp=make_comp())
                rows.append(dict(dataset=t, scorer=sc, K=K, policy=lab, theta=np.nan, cost=tx_cost(b, r, y1, A["at"]).sum()))
TC = pd.DataFrame(rows); TC.to_csv(f"{DIRS['output']}/tuned_and_decomposition.csv", index=False)

W = TC.pivot_table(index=["dataset", "scorer", "K"], columns="policy", values="cost").reset_index()
comp_cols = ["P1", "P1-c (cost-tuned threshold)", "P2", "P3", "P4", "P5", "P5-tuned", "P6", "P6-tuned"]
W["best_competitor_tuned"] = W[comp_cols].min(1)
W["P7_vs_best_tuned_pct"] = 100 * (W.best_competitor_tuned - W.P7) / W.best_competitor_tuned
W["P7_vs_P6tuned_pct"] = 100 * (W["P6-tuned"] - W.P7) / W["P6-tuned"]
# 2x2 decomposition of P6 - P7 (fixed theta_H = 0.9): block-rule effect and review-selection effect
W["gap_P6_P7"] = W.P6 - W.P7
W["effect_block"] = 0.5 * ((W.P6 - W.A2) + (W.A1 - W.P7))      # fixed theta_H -> expected-cost block
W["effect_review"] = 0.5 * ((W.P6 - W.A1) + (W.A2 - W.P7))     # fill by p*a -> gain-filtered reviews
W.to_csv(f"{DIRS['output']}/tuned_and_decomposition_wide.csv", index=False)

for t in DEC:
    w = W[W.dataset == t]; m = w[w.K == MID_K[t]]
    note(f"{t}_tuned_P7_lowest_cells", int((w.P7_vs_best_tuned_pct >= -1e-9).sum()), "tuned_and_decomposition_wide.csv")
    note(f"{t}_tuned_cells", int(len(w)), "tuned_and_decomposition_wide.csv")
    note(f"{t}_tuned_P7_vs_best_median_pct", float(w.P7_vs_best_tuned_pct.median()), "tuned_and_decomposition_wide.csv")
    note(f"{t}_P7_vs_P6tuned_median_pct", float(w.P7_vs_P6tuned_pct.median()), "tuned_and_decomposition_wide.csv")
    for sc in ["LightGBM", "FTT_ens"]:
        r = m[m.scorer == sc].iloc[0]
        for k_ in ["gap_P6_P7", "effect_block", "effect_review"]:
            note(f"{t}_decomp_{sc}_K{MID_K[t]}_{k_}", float(r[k_]), "tuned_and_decomposition_wide.csv")
    print(f"\n== {DS_NAME[t]}: P7 vs best tuned competitor (%), median {w.P7_vs_best_tuned_pct.median():.1f}, "
          f"P7 lowest in {(w.P7_vs_best_tuned_pct >= -1e-9).sum()}/{len(w)} cells")
    print(m[["scorer", "P6", "P6-tuned", "A1", "A2", "P7", "gap_P6_P7", "effect_block", "effect_review"]].round(0))
print(f"{time.time()-t0:.0f} s")

## Step 23: Alternative cost models
Revision analysis of the cost assumptions of Eq. (3), at the middle budget,
offline selection, isotonic calibration, all scorers. Each scenario changes the
realised cost and is also given to the policies that use costs (P2, P6, P7):
- **Partial recovery:** only a share `rho` in `RECOVERY` of a missed fraud
  amount is lost (chargebacks, recoveries).
- **Amount-dependent blocking cost:**
  `C_blk,i = C_BLK * (CBLK_FIXED + (1 - CBLK_FIXED) * a_i / median legitimate validation amount)`,
  equal to `C_BLK` at the median amount.
- **Amount-dependent analyst accuracy:** `q_i = Q_BY_AMOUNT[0]` at or below the
  median validation fraud amount and `Q_BY_AMOUNT[1]` above it. P7 assumes
  `q = 1`; P7-q uses `q_i`.
For each scenario the Holm-corrected block bootstrap of Step 21 compares P7
(and P7-q) with every other policy.

In [ ]:
t0 = time.time(); rows, brows = [], []
for t in DEC:
    D = DEC[t]; yv, y1 = D["yva"], D["yte"] == 1; K = MID_K[t]
    av_, at_ = D["amt_va"].astype(float), D["amt_te"].astype(float)
    med_leg = np.median(av_[yv == 0]); med_fr = np.median(av_[yv == 1])
    scen = {f"recovery_{r}": dict(a_scale=r) for r in RECOVERY}
    scen["amount_dependent_Cblk"] = dict(cblk=lambda a: C_BLK * (CBLK_FIXED + (1 - CBLK_FIXED) * a / med_leg))
    scen["amount_dependent_q"] = dict(q=lambda a: np.where(a <= med_fr, Q_BY_AMOUNT[0], Q_BY_AMOUNT[1]))
    for sn, S in scen.items():
        for sc in SCORERS:
            A = arrays(t, sc); r_ = S.get("a_scale", 1.0)
            av, at = A["av"] * r_, A["at"] * r_
            cb_va = S["cblk"](A["av"]) if "cblk" in S else C_BLK; cb_te = S["cblk"](A["at"]) if "cblk" in S else C_BLK
            q_te = S["q"](A["at"]) if "q" in S else 1.0
            pols = POLICIES + ([PROPOSED + "_q"] if "q" in S else [])
            costs = []
            for p in pols:
                name, qp = (PROPOSED, q_te) if p.endswith("_q") else (p, 1.0)
                if p.endswith("_q"):   # P7-q needs q per transaction on validation too (only the offline mode is used here)
                    b, r = decide_arr(name, A["pv"], A["pt"], av, at, A["th"], K, "offline", c_blk=cb_te, q=qp)
                else:
                    b, r = decide_arr(name, A["pv"], A["pt"], av, at, A["th"], K, "offline", c_blk=cb_te, cblk_va=cb_va)
                costs.append(tx_cost(b, r, y1, at, c_blk=cb_te, q=q_te))
            tot = {p: c.sum() for p, c in zip(pols, costs)}
            for p in pols: rows.append(dict(dataset=t, scenario=sn, scorer=sc, policy=p, cost=tot[p]))
            for ref in [PROPOSED] + ([PROPOSED + "_q"] if "q" in S else []):
                for b_ in block_boot(costs, pols.index(ref)):
                    brows.append(dict(dataset=t, scenario=sn, scorer=sc, ref=ref, competitor=pols[b_.pop("col")], **b_))
CM = pd.DataFrame(rows); CM.to_csv(f"{DIRS['output']}/cost_model_scenarios.csv", index=False)
CB = pd.DataFrame(brows); CB["holm_reject"] = False
for key, g in CB.groupby(["dataset", "scenario", "ref"]):
    CB.loc[g.index, "holm_reject"] = holm(g.p.values)
CB["verdict_holm"] = np.where(CB.holm_reject, np.where(CB["diff"] > 0, "better", "worse"), "ns")
CB.to_csv(f"{DIRS['output']}/cost_model_scenarios_bootstrap.csv", index=False)

Wc = CM.pivot_table(index=["dataset", "scenario", "scorer"], columns="policy", values="cost").reset_index()
others = [p for p in POLICIES if p != PROPOSED]
Wc["best_other"] = Wc[others].idxmin(1); Wc["best_other_cost"] = Wc[others].min(1)
Wc["P7_vs_best_pct"] = 100 * (Wc.best_other_cost - Wc[PROPOSED]) / Wc.best_other_cost
Wc["P7_vs_P1_pct"] = 100 * (Wc.binary_F1 - Wc[PROPOSED]) / Wc.binary_F1
Wc.to_csv(f"{DIRS['output']}/cost_model_scenarios_wide.csv", index=False)
for (t, sn), g in Wc.groupby(["dataset", "scenario"]):
    note(f"{t}_scen_{sn}_P7_vs_best_median_pct", float(g.P7_vs_best_pct.median()), "cost_model_scenarios_wide.csv")
    note(f"{t}_scen_{sn}_P7_vs_P1_median_pct", float(g.P7_vs_P1_pct.median()), "cost_model_scenarios_wide.csv")
    note(f"{t}_scen_{sn}_P7_lowest_scorers", int((g.P7_vs_best_pct >= -1e-9).sum()), "cost_model_scenarios_wide.csv")
for (t, sn, ref), g in CB.groupby(["dataset", "scenario", "ref"]):
    v = g.verdict_holm.value_counts()
    for vd in ["better", "ns", "worse"]:
        note(f"{t}_scen_{sn}_{'P7q' if ref.endswith('_q') else 'P7'}_holm_{vd}", int(v.get(vd, 0)), "cost_model_scenarios_bootstrap.csv")
print(Wc[["dataset", "scenario", "scorer", "best_other", "P7_vs_best_pct", "P7_vs_P1_pct"]].round(1).to_string())
print(f"{time.time()-t0:.0f} s")

## Step 24: Distribution shift between validation and test
Revision analysis of the deployable modes under shift.
- **Natural drift.** The test window follows the validation window in time.
  For every dataset and scorer we report the fraud rate of both windows, the
  mean calibrated probability on the test window (calibration-in-the-large) and
  the review volume that the validation threshold of P7 would select without a
  cap, relative to `K`.
- **Prevalence shift.** The calibrator and the thresholds stay fixed while the
  fraud prevalence of the test window is multiplied by the factors in
  `PREV_FACTORS`: frauds are subsampled (factor < 1) or resampled with
  replacement (factor > 1), legitimate transactions are kept, and the time
  order is preserved. `N_SHIFT` resamples per factor; the budget `K` is the
  middle value and the review threshold is learned on validation as in
  Eq. (8).

In [ ]:
t0 = time.time(); drows, srows = [], []
rng = np.random.default_rng(2027)
for t in DEC:
    D = DEC[t]; yv, yt = D["yva"], D["yte"]; K = MID_K[t]
    for sc in SCORERS:
        A = arrays(t, sc)
        _, cand_v, prio_v = components(PROPOSED, A["pv"], A["av"], A["th"])
        tau = kth_value(prio_v, cand_v, round(K * len(yv) / len(yt)))
        _, cand_t, prio_t = components(PROPOSED, A["pt"], A["at"], A["th"])
        drows.append(dict(dataset=t, scorer=sc, fraud_rate_val=yv.mean(), fraud_rate_test=yt.mean(),
                          mean_p_test=A["pt"].mean(), P7_uncapped_reviews_over_K=(cand_t & (prio_t >= tau)).sum() / K))
        pos, neg = np.where(yt == 1)[0], np.where(yt == 0)[0]
        for f in PREV_FACTORS:
            for rep in range(N_SHIFT):
                fr = rng.choice(pos, int(round(f * len(pos))), replace=f > 1)
                idx = np.sort(np.concatenate([neg, fr]), kind="stable"); y1 = yt[idx] == 1
                for m in ["offline", "online_capped"]:
                    for p in POLICIES if m == "offline" else BUDGETED:
                        b, r = decide_arr(p, A["pv"], A["pt"][idx], A["av"], A["at"][idx], A["th"], K, m)
                        srows.append(dict(dataset=t, scorer=sc, factor=f, rep=rep, mode=m, policy=p,
                                          cost=tx_cost(b, r, y1, A["at"][idx]).sum(), reviews=int(r.sum())))
DR = pd.DataFrame(drows); DR.to_csv(f"{DIRS['output']}/drift_summary.csv", index=False)
SH = pd.DataFrame(srows); SH.to_csv(f"{DIRS['output']}/prevalence_shift_runs.csv", index=False)

Ws = SH.pivot_table(index=["dataset", "scorer", "factor", "rep", "mode"], columns="policy", values="cost").reset_index()
others = [p for p in POLICIES if p != PROPOSED]
Ws["best_other"] = Ws[[c for c in others if c in Ws]].min(1)
Ws["P7_vs_best_pct"] = 100 * (Ws.best_other - Ws[PROPOSED]) / Ws.best_other
Ws["P7_vs_P6_pct"] = 100 * (Ws.threeway_expected_loss - Ws[PROPOSED]) / Ws.threeway_expected_loss
S2 = Ws.groupby(["dataset", "factor", "mode"]).agg(P7_vs_best_median=("P7_vs_best_pct", "median"),
        P7_vs_P6_median=("P7_vs_P6_pct", "median"), P7_best_share=("P7_vs_best_pct", lambda v: (v >= -1e-9).mean())).reset_index()
on = SH[SH["mode"] == "online_capped"].merge(SH[SH["mode"] == "offline"], on=["dataset", "scorer", "factor", "rep", "policy"], suffixes=("_on", "_off"))
on["online_change_pct"] = 100 * (on.cost_on - on.cost_off) / on.cost_off
S3 = on.groupby(["dataset", "factor", "policy"]).online_change_pct.median().unstack()
S2.to_csv(f"{DIRS['output']}/prevalence_shift_summary.csv", index=False); S3.to_csv(f"{DIRS['output']}/prevalence_shift_online_change.csv")
for _, r in S2.iterrows():
    k = f"{r.dataset}_shift{r.factor}_{r['mode']}"
    note(k + "_P7_vs_best_median_pct", r.P7_vs_best_median, "prevalence_shift_summary.csv")
    note(k + "_P7_vs_P6_median_pct", r.P7_vs_P6_median, "prevalence_shift_summary.csv")
    note(k + "_P7_best_share", r.P7_best_share, "prevalence_shift_summary.csv")
for (t, f), r in S3.iterrows():
    for p in [PROPOSED, "threeway_expected_loss"]:
        note(f"{t}_shift{f}_online_change_median_pct_{POL_SHORT[p]}", float(r[p]), "prevalence_shift_online_change.csv")
for _, r in DR.iterrows():
    note(f"{r.dataset}_drift_{r.scorer}_mean_p_test", r.mean_p_test, "drift_summary.csv")
    note(f"{r.dataset}_drift_{r.scorer}_P7_uncapped_over_K", r.P7_uncapped_reviews_over_K, "drift_summary.csv")
for t in DEC:
    note(f"{t}_fraud_rate_val", float(DEC[t]["yva"].mean()), "drift_summary.csv"); note(f"{t}_fraud_rate_test", float(DEC[t]["yte"].mean()), "drift_summary.csv")
print(DR.round(4)); print(S2.round(2)); print(S3.round(1))
print(f"{time.time()-t0:.0f} s")

## Step 25: Additional chronological windows (IEEE-CIS, LightGBM)
Revision analysis of the dependence on one test window. The IEEE-CIS data are
split into three rolling-origin windows `ROLL_WINDOWS`, each with 15%
validation and 15% test after an expanding training partition; W3 is the main
split. For W1 and W2 a LightGBM model with the settings of Step 4 is trained on
the training part and early-stopped on the validation part; calibration,
thresholds, policies, budget modes and the Holm-corrected block bootstrap are
those of the main analysis. The features are the encoded matrix of Step 3; its
frequency encoding uses training-partition frequencies of the main split
(label-free counts).

In [ ]:
t0 = time.time()
T_ = np.load(f"{DIRS['dataset']}/ieee_tree.npz"); Mt = np.load(f"{DIRS['dataset']}/ieee_meta.npz", allow_pickle=True)
Xall = np.vstack([T_["Xtr"], T_["Xva"], T_["Xte"]]); yall = np.concatenate([T_["ytr"], T_["yva"], T_["yte"]]).astype(int)
amt_all = Mt["amt"].astype(float); day_all = ((Mt["t"] - Mt["t"].min()) // 86400).astype(int)
assert len(amt_all) == len(yall) and (Mt["y"].astype(int) == yall).all()      # same time order as the tree matrices
N = len(yall); rows, brows = [], []
for w, (e_tr, e_va, e_te) in ROLL_WINDOWS.items():
    s_tr, s_va, s_te = slice(0, int(e_tr * N)), slice(int(e_tr * N), int(e_va * N)), slice(int(e_va * N), int(e_te * N))
    yv, yt = yall[s_va], yall[s_te]
    if w == "W3":
        pv_raw, pt_raw = load_pred("ieee", "LightGBM"); assert len(pv_raw) == len(yv) and len(pt_raw) == len(yt)
    else:
        m = lgb.LGBMClassifier(n_estimators=3000, num_leaves=31, learning_rate=0.05, random_state=42, n_jobs=-1, verbose=-1)
        m.fit(Xall[s_tr], yall[s_tr], eval_set=[(Xall[s_va], yv)], eval_metric="average_precision",
              callbacks=[lgb.early_stopping(100, verbose=False)])
        pv_raw, pt_raw = m.predict_proba(Xall[s_va])[:, 1], m.predict_proba(Xall[s_te])[:, 1]
    pv, pt = iso(pv_raw, yv, pv_raw), iso(pv_raw, yv, pt_raw); th = f1_threshold(yv, pv)
    av, at, per_va, per_te = amt_all[s_va], amt_all[s_te], day_all[s_va], day_all[s_te]; y1 = yt == 1
    rows.append(dict(window=w, part="data", n_test=len(yt), fraud_rate_test=yt.mean(), AUPRC=average_precision_score(yt, pt_raw),
                     test_start_day=int(per_te.min()), test_end_day=int(per_te.max())))
    for K in K_GRID["ieee"]:
        for md in ["offline", "online_capped", "period_adaptive"]:
            pols = POLICIES if md == "offline" else BUDGETED
            costs = [tx_cost(*decide_arr(p, pv, pt, av, at, th, K, md, per_va, per_te), y1, at) for p in pols]
            for p, c in zip(pols, costs): rows.append(dict(window=w, part="cost", K=K, mode=md, policy=p, cost=c.sum()))
            for b_ in block_boot(costs, pols.index(PROPOSED)):
                brows.append(dict(window=w, K=K, mode=md, competitor=pols[b_.pop("col")], **b_))
    print(f"{w}: test fraud rate {yt.mean():.4f} | AUPRC {average_precision_score(yt, pt_raw):.3f} | {time.time()-t0:.0f} s")
RW = pd.DataFrame(rows); RW.to_csv(f"{DIRS['output']}/rolling_windows.csv", index=False)
RB = pd.DataFrame(brows); RB["holm_reject"] = False
for key, g in RB.groupby(["window", "mode"]):
    RB.loc[g.index, "holm_reject"] = holm(g.p.values)
RB["verdict_holm"] = np.where(RB.holm_reject, np.where(RB["diff"] > 0, "better", "worse"), "ns")
RB.to_csv(f"{DIRS['output']}/rolling_windows_bootstrap.csv", index=False)

Rc = RW[RW.part == "cost"].pivot_table(index=["window", "K", "mode"], columns="policy", values="cost").reset_index()
for p in POLICIES:
    if p != PROPOSED and p in Rc: Rc[f"red_vs_{POL_SHORT[p]}"] = 100 * (Rc[p] - Rc[PROPOSED]) / Rc[p]
Rc.to_csv(f"{DIRS['output']}/rolling_windows_wide.csv", index=False)
mid = Rc[(Rc.K == MID_K["ieee"]) & (Rc["mode"] == "offline")]
for _, r in mid.iterrows():
    for c in ["red_vs_P1", "red_vs_P2", "red_vs_P6"]:
        note(f"ieee_roll_{r.window}_{c}_pct", float(r[c]), "rolling_windows_wide.csv")
for _, r in RW[RW.part == "data"].iterrows():
    note(f"ieee_roll_{r.window}_AUPRC", float(r.AUPRC), "rolling_windows.csv"); note(f"ieee_roll_{r.window}_fraud_rate", float(r.fraud_rate_test), "rolling_windows.csv")
for (w, md), g in RB.groupby(["window", "mode"]):
    v = g.pivot_table(index="competitor", columns="verdict_holm", values="diff", aggfunc="count", fill_value=0)
    for comp in v.index:
        for vd in v.columns: note(f"ieee_roll_{w}_{md}_{POL_SHORT[comp]}_{vd}", int(v.loc[comp, vd]), "rolling_windows_bootstrap.csv")
    print(f"-- {w} {md}"); print(v)
print(mid[["window"] + [c for c in mid if c.startswith("red_vs")]].round(1))
print(f"{time.time()-t0:.0f} s")

## Step 26: Revision tables
Tables for the revised manuscript, in CSV and LaTeX (`output/tables/`):
- `table5b_block_holm`: Table 5 recomputed with the moving-block bootstrap and
  the Holm correction (better / not significant / worse per competitor).
- `table7_effect_sizes`: relative cost reduction of P7 with 95% block-bootstrap
  intervals for the primary comparisons (P2, P6), middle budget.
- `table8_robustness`: one row per revision analysis (tuned competitors,
  decomposition, alternative cost models, prevalence shift, additional
  windows) with the median reduction of P7 relative to the best competitor.

In [ ]:
rows = []
for p in POLICIES[:-1]:
    r_ = {"Competitor": POL_NAME[p]}
    for t in DEC:
        for lab, md in [("Off.", "offline"), ("On.", "online_capped"), ("Per.", "period_adaptive")]:
            b = BB[(BB.dataset == t) & (BB.competitor == p) & (BB["mode"] == md)]
            r_[f"{DS_NAME[t]} {lab}"] = "--" if b.empty else "/".join(str(int((b.verdict_holm == v).sum())) for v in ["better", "ns", "worse"])
    rows.append(r_)
to_tex(pd.DataFrame(rows), "table5b_block_holm",
       "Moving-block bootstrap of the budgeted Bayes policy (P7) against each competitor with the Holm correction within each dataset and budget mode: number of cells (scorer $\\times$ budget) in which P7 is significantly cheaper / not significantly different / significantly more expensive (family-wise level 0.05, 10,000 resamples of 50 blocks). Off.: offline; On.: window budget enforced on arrival with a hard cap; Per.: per-period capacity with the adaptive threshold.",
       "tab:boot", "l" + "r" * 9)

rows = []
for t in DEC:
    for sc in SCORERS:
        r_ = {"Dataset": DS_NAME[t] if sc == SCORERS[0] else "", "Scorer": SC_NAME[sc]}
        for md, lab in [("offline", "offline"), ("online_capped", "online")]:
            for comp in PRIMARY:
                e = ES[(ES.dataset == t) & (ES.scorer == sc) & (ES.competitor == comp) & (ES["mode"] == md)]
                if e.empty: continue
                e = e.iloc[0]
                star = "" if e.verdict_holm == "better" else ("$^{\\dagger}$" if e.verdict_holm == "ns" else "$^{\\ddagger}$")
                r_[f"{POL_SHORT[comp]} {lab}"] = f"{e.rel_pct:.1f} [{e.rel_lo:.1f}, {e.rel_hi:.1f}]{star}"
        rows.append(r_)
to_tex(pd.DataFrame(rows), "table7_effect_sizes",
       "Relative cost reduction (\\%) of P7 at the middle budget against the pre-specified primary competitors P2 and P6, with 95\\% moving-block bootstrap intervals; offline selection and window budget enforced on arrival. $^{\\dagger}$ not significant and $^{\\ddagger}$ significantly worse after the Holm correction; all other entries are significantly better.",
       "tab:effects", "llrrr")

def med(s): return f"{np.median(s):.1f}"
rows = []
for t in DEC:
    w = W[W.dataset == t]; wc = Wc[Wc.dataset == t]; s2 = S2[S2.dataset == t]
    rows += [dict(Analysis="Tuned thresholds (P1-c, P5/P6 tuned): P7 vs best", Dataset=DS_NAME[t], Value=med(w.P7_vs_best_tuned_pct),
                  Detail=f"P7 lowest in {(w.P7_vs_best_tuned_pct >= -1e-9).sum()}/{len(w)} cells")]
    m = w[w.K == MID_K[t]]
    rows += [dict(Analysis="Decomposition of P6 - P7 (block / review), LightGBM", Dataset=DS_NAME[t],
                  Value=f"{m[m.scorer == 'LightGBM'].effect_block.iloc[0]:,.0f} / {m[m.scorer == 'LightGBM'].effect_review.iloc[0]:,.0f}",
                  Detail=f"gap {m[m.scorer == 'LightGBM'].gap_P6_P7.iloc[0]:,.0f}")]
    for sn, g in wc.groupby("scenario"):
        rows.append(dict(Analysis=f"Cost model: {sn.replace('_', ' ')}", Dataset=DS_NAME[t], Value=med(g.P7_vs_best_pct),
                         Detail=f"P7 lowest for {(g.P7_vs_best_pct >= -1e-9).sum()}/{len(g)} scorers"))
    for (f, md), g in s2.groupby(["factor", "mode"]):
        rows.append(dict(Analysis=f"Prevalence x{f:g}, {md.replace('_', ' ')}", Dataset=DS_NAME[t], Value=f"{g.P7_vs_best_median.iloc[0]:.1f}",
                         Detail=f"P7 lowest in {100 * g.P7_best_share.iloc[0]:.0f}% of resamples"))
mid_r = Rc[(Rc.K == MID_K["ieee"]) & (Rc["mode"] == "offline")]
for _, r in mid_r.iterrows():
    rows.append(dict(Analysis=f"Window {r.window} (LightGBM): P7 vs P1 / P2 / P6", Dataset="IEEE-CIS",
                     Value=f"{r.red_vs_P1:.1f} / {r.red_vs_P2:.1f} / {r.red_vs_P6:.1f}", Detail=""))
to_tex(pd.DataFrame(rows), "table8_robustness",
       "Revision analyses at the middle budget (isotonic calibration). Value: median cost reduction (\\%) of P7 relative to the cheapest competitor across scorers (or resamples), unless stated otherwise; decomposition in cost units.",
       "tab:robust", "llrl")
print(pd.read_csv(f"{TB}/table5b_block_holm.csv").to_string()); print(pd.read_csv(f"{TB}/table7_effect_sizes.csv").to_string())
print(pd.read_csv(f"{TB}/table8_robustness.csv").to_string())

## Step 27: Export
Everything needed for the manuscript is bundled in one archive:
`fig/` (main and extra figures), `output/` (all result CSVs, tables in CSV and
LaTeX, `manuscript_numbers.csv`). Prediction arrays and model weights remain in
the notebook output.

In [ ]:
import zipfile
pd.DataFrame(NUMBERS).to_csv(f"{DIRS['output']}/manuscript_numbers.csv", index=False)
files = sorted(glob.glob(f"{DIRS['fig']}/**/*.*", recursive=True) + glob.glob(f"{DIRS['output']}/**/*.csv", recursive=True)
               + glob.glob(f"{DIRS['output']}/**/*.tex", recursive=True) + glob.glob(f"{DIRS['output']}/*.json"))
with zipfile.ZipFile(f"{WORK}/paper_package.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in files: z.write(f, arcname=os.path.relpath(f, WORK))
print(f"{len(files)} files -> {WORK}/paper_package.zip ({os.path.getsize(f'{WORK}/paper_package.zip')/1e6:.1f} MB)")
print(f"{len(NUMBERS)} numbers in manuscript_numbers.csv")